# 🎓 Diplomatura en Desarrollo de Aplicaciones con LLMs y Modelos Generativos
## Laboratorio 3: Sistemas RAG (Consulta Documental Fundamentada)

---
### 📌 Encuadre Académico y Esquema de Evaluación
* **Asignatura:** Desarrollo de Aplicaciones con LLMs y Modelos Generativos (UNC).
* **Módulo:** Práctica Guiada — Bloque de Laboratorio (2 horas sincrónicas).
* **Esquema de Evaluación:** Este laboratorio conforma la **Parte 1 del Trabajo Práctico 2 (TP2)**. Junto con el Laboratorio 4 (Uso de Herramientas, Validación y Agentes), integra la entrega final del programa.
* **Criterio de Aprobación:** Ejecutar los 4 experimentos y completar las reflexiones en los recuadros editables (`✍️ Tu análisis aquí:`) al pie de cada experimento.

---
### 🧭 Cómo está construido este laboratorio

Cada experimento está diseñado como un **ensayo controlado con veredicto automático**. En vez de pedirte que leas unos textos y opines si el resultado "se ve mejor", el notebook:

1. Te muestra una configuración que **falla de manera inequívoca**,
2. otra que **funciona correctamente**,
3. y un **barrido gradual** entre ambas para que veas *dónde exactamente* está la frontera.

El diagnóstico lo emite el código —contando cláusulas perdidas, comparando contra la respuesta correcta anotada, detectando cifras inventadas—, **no tu impresión al leer la salida**. Si un resultado te parece ambiguo, esa ambigüedad es un dato del experimento y merece estar en tu análisis.

> ⚠️ **Sobre el corpus:** el manual de pólizas que vamos a indexar está **deliberadamente minado**. Contiene nueve códigos de cláusula casi idénticos entre sí, cuatro franquicias distintas en cuatro artículos distintos, y cuatro reglas cuya excepción vive en el párrafo siguiente. No es un documento "difícil" por accidente: es un set de pruebas donde los errores clásicos de RAG se vuelven visibles y medibles.

---
### 🎯 Objetivos de Aprendizaje Operativo
Al finalizar esta sesión serás capaz de:
1. **Medir la calidad de una segmentación** con criterios objetivos —integridad normativa y cortes mutilados— en lugar de inspeccionar fragmentos a ojo, y demostrar cómo un defecto de *chunking* se transforma en una respuesta incorrecta al cliente.
2. **Calibrar el solape (*Chunk Overlap*)** identificando el punto exacto en que las excepciones legales dejan de perderse, y justificar por qué la franja del 10–20% es la recomendación estándar.
3. **Contrastar los tres motores de recuperación** —denso (*embeddings*), disperso (**BM25**) e **híbrido (RRF)**— identificando el punto ciego propio de cada uno y explicando por qué la búsqueda híbrida es el estándar de producción.
4. **Implementar generación fundamentada** con citas verificables y regla de abstención, y **auditarla automáticamente** detectando cifras inventadas, cifras trasladadas de otro artículo y citas a fuentes inexistentes.
5. **Calibrar un filtro de Corrective RAG (CRAG)** construyendo su matriz de confusión sobre un set etiquetado, y reconocer **qué clase de fallas el umbral no puede detectar**.
6. **Interpretar la telemetría de inferencia** (TTFT, TPOT, EERL y economía de tokens) como insumo de decisiones de arquitectura y costo.

---
### ⚙️ Configuración Obligatoria del Entorno en Google Colab
Para garantizar la inferencia acelerada sin costo alguno:
1. En la barra superior de menús, haz clic en **Entorno de ejecución** (*Runtime*).
2. Selecciona **Cambiar tipo de entorno de ejecución** (*Change runtime type*).
3. En **Acelerador de hardware**, verifica que esté seleccionada la opción **GPU T4**.
4. Haz clic en **Guardar**.

> ⏱️ **Ruta rápida (si vas justo de tiempo):** ejecutá el Bloque 0 completo y después sólo las celdas marcadas con 🚀. Las celdas 🧪 son de exploración libre y podés dejarlas para después de clase.

## 🛠️ Bloque 0: Preparación del Entorno y Motor Local

Preparamos el motor de inferencia local. Usamos **`llama-cpp-python`** compilado con soporte CUDA para ejecutar un modelo de pesos abiertos en formato **GGUF cuantizado a 4 bits**, junto con un modelo liviano de representaciones vectoriales densas multilingüe. Todo corre en la VRAM de la GPU T4, con costo cero y sin que un solo fragmento del documento salga del entorno.

* **Motor Generador (LLM):** seleccionable en la celda `0.2`. Por defecto `Qwen 2.5 3B Instruct` en formato `GGUF (Q4_K_M)` (~2.0 GB de VRAM).
* **Motor de Embeddings:** `paraphrase-multilingual-MiniLM-L12-v2` (~400 MB de VRAM, optimizado para español). **No es seleccionable**: cambiarlo obligaría a reindexar el corpus y movería los umbrales de similitud calibrados en el Experimento 4.
* **Base Documental:** *Manual de Políticas de Cobertura, Siniestros y Cláusulas Operativas (Edición 2024)* de la aseguradora Vertex Horizon Seguros — 14 artículos con distractores deliberados.

> 💡 **Nota sobre Modelos:** Si bien se recomienda realizar la experimentación inicial con el modelo predeterminado, se invita a los estudiantes a **experimentar libremente con los otros 4 modelos de la lista** (Llama 3.2, Gemma 2, Gemma 4 y DeepSeek) para comparar sus estilos y métricas. Esta exploración adicional es opcional y no forma parte de los criterios de evaluación obligatorios.

> ⚠️ **Advertencia propia de este laboratorio:** los contrastes de los cuatro experimentos fueron **calibrados con el modelo predeterminado**. Este laboratorio depende de que el modelo obedezca instrucciones difíciles —abstenerse cuando el dato no está, citar el fragmento exacto— y no todos los modelos de 2-3 B lo logran con la misma consistencia. Si cambiás de modelo y un contraste no sale como el texto anuncia, **eso no es un error del notebook: es un hallazgo**, y la celda `0.5` es la que te dice qué se movió. La familia Gemma, además, no admite el rol `system`: el laboratorio lo detecta solo y funde las instrucciones dentro del mensaje de usuario.

> 🔒 **Por qué importa que corra local:** la privacidad no es un detalle del ejercicio. Como se discutió en la teoría, los *embeddings* pueden invertirse parcialmente para reconstruir entidades del texto original, y las bases vectoriales guardan el texto plano junto al vector. Un corpus de pólizas reales enviado a un servicio de terceros es una transferencia de datos sensibles, no una llamada a una API.

In [ ]:
#@title 0.1 Instalación silenciosa de dependencias optimizadas para GPU T4 { display-mode: "form" }
import sys

print("⏳ Instalando dependencias con aceleración CUDA precompilada y librerías vectoriales...")

try:
    # llama-cpp-python precompilado para CUDA 12.2 en Google Colab
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
      --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q

    # Librerías de soporte
    !pip install huggingface_hub sentence-transformers ipywidgets pandas scikit-learn -q
    print("✅ Entorno preparado (llama-cpp-python, sentence-transformers, huggingface_hub, ipywidgets).")
except Exception as e:
    print(f"❌ Error durante la instalación de paquetes: {e}")

In [ ]:
#@title 0.2 Selección y montaje del modelo en GPU VRAM { run: "auto" }
import os, time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer

#@markdown Selecciona el modelo generador que deseas cargar en la GPU T4:
modelo_seleccionado = "Qwen 2.5 3B Instruct (Alibaba)" #@param ["Qwen 2.5 3B Instruct (Alibaba)", "Llama 3.2 3B Instruct (Meta)", "Gemma 2 2B Instruct (Google)", "Gemma 4 E2B Instruct (Experimental)", "DeepSeek-Coder-V2 3B (DeepSeek)"] {run: "auto"}

modelos_disponibles = {
    "Qwen 2.5 3B Instruct (Alibaba)": {
        "repo_id": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "filename": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "Llama 3.2 3B Instruct (Meta)": {
        "repo_id": "bartowski/Llama-3.2-3B-Instruct-GGUF",
        "filename": "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
    },
    "Gemma 2 2B Instruct (Google)": {
        "repo_id": "bartowski/gemma-2-2b-it-GGUF",
        "filename": "gemma-2-2b-it-Q4_K_M.gguf"
    },
    "Gemma 4 E2B Instruct (Experimental)": {
        "repo_id": "unsloth/gemma-4-E2B-it-GGUF",
        "filename": "gemma-4-E2B-it-Q4_K_M.gguf"
    },
    "DeepSeek-Coder-V2 3B (DeepSeek)": {
        "repo_id": "bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF",
        "filename": "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"
    }
}

config = modelos_disponibles[modelo_seleccionado]
print(f"⏳ Iniciando proceso para: {modelo_seleccionado}...")

try:
    # ---- 1. Modelo GENERADOR (el que cambia con el selector) ----
    model_path = hf_hub_download(
        repo_id=config["repo_id"],
        filename=config["filename"],
        local_dir="./modelos"
    )
    print(f"📦 Pesos listos en: {model_path}")
    print("⏳ Cargando el LLM en VRAM (offloading total: n_gpu_layers=-1)...")

    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1,      # todas las capas a la GPU
        n_ctx=4096,           # ventana amplia: prompt + evidencia recuperada
        verbose=False
    )
    print(f"✅ {modelo_seleccionado} montado en GPU.")

    # ---- 2. Modelo de EMBEDDINGS (fijo: define el índice, no la generación) ----
    # El selector de arriba cambia SOLO el generador. El recuperador usa siempre el
    # mismo modelo de embeddings, porque cambiarlo obligaría a reindexar el corpus
    # y movería todos los umbrales de similitud calibrados en el Experimento 4.
    print("⏳ Montando el modelo de embeddings multilingüe...")
    embedding_model = SentenceTransformer(
        "paraphrase-multilingual-MiniLM-L12-v2",
        device="cuda"
    )
    print("✅ Motor de embeddings operativo (paraphrase-multilingual-MiniLM-L12-v2).")
except Exception as e:
    print(f"❌ Error al cargar los modelos en GPU: {str(e)}")

### 🧨 Qué hay dentro del corpus (y por qué está minado)

La celda que sigue define el documento sobre el que trabaja todo el laboratorio: un manual de pólizas de 14 artículos. No es un texto cualquiera: está **construido para que los errores clásicos de RAG se vuelvan visibles y medibles**. Tiene tres trampas deliberadas.

**1. Cada regla tiene su excepción en el párrafo siguiente.** Este es el ejemplo del Artículo 5, y es el que va a fallar en el Experimento 1:

| | Texto en el manual |
| :--- | :--- |
| ✅ **La regla** | *"…la compania indemnizara el valor de reposicion vigente… Sobre el valor de origen se aplica una **depreciacion del 25% anual** acumulada segun la antiguedad del rodado."* |
| ⛔ **La excepción**, 69 caracteres después | *"**EXCLUSION EXC-114:** Se deniega la cobertura de robo total cuando la unidad no contaba con el **dispositivo de rastreo satelital** homologado… al momento de la ocurrencia del hecho."* |

Si el fragmento que llega al modelo **corta entre las dos**, el sistema le informa al asegurado cómo se calcula su indemnización y **omite que sin rastreador no cobra nada**. No hay error, no hay excepción en el log: hay una promesa de pago que la compañía no está obligada a cumplir.

Hay **cuatro pares así**, separados por entre 69 y 97 caracteres:

| Artículo | La regla promete… | …y la excepción la anula |
| :--- | :--- | :--- |
| Art. 1 · Climático | liquidar el 100% del peritaje | `CLUST-90`: nada en zona rural sin diques homologados |
| Art. 3 · Rescisión | descontar sólo un 8% de arancel | `RES-22`: dentro de 30 días, devolución íntegra |
| Art. 5 · Robo total | valor de reposición − 25% anual | `EXC-114`: sin rastreo satelital activo, no hay cobertura |
| Art. 6 · Agro | hasta 40 hectáreas por campaña | `ANX-7B`: exige declaración de siembra anticipada |

**2. Nueve códigos de cláusula con la misma forma** (`CLUST-90`, `RES-22`, `EXC-114`, `ANX-7B`, `INC-57`, `DIC-45`, `MER-08`, `PER-31`, `RCT-19`). Para un modelo de *embeddings* todos "significan" prácticamente lo mismo: un código de una póliza. Es la trampa del Experimento 2.

**3. Cuatro franquicias distintas en cuatro artículos distintos** (150, 300 y 90 USD… y ninguna para el robo de vehículos). Es la trampa del Experimento 3: cuando se pregunta por la franquicia de robo, el modelo tiene tres cifras a mano y **todas son incorrectas** para esa pregunta.

---

### 📐 Por qué cada fragmento guarda `ini` y `fin`

El segmentador no devuelve sólo texto: cada fragmento recuerda **en qué tramo exacto del documento original empieza y termina** (`ini` y `fin`). Así se ve con el propio Artículo 5: la regla empieza en el carácter 3149 y la excepción termina en el 3235 — hay que cubrir esos 86 caracteres **dentro de un mismo fragmento** para que el sistema no pueda prometer sin ver la condición.

| Fragmento | `ini` | `fin` | ¿Cubre el rango 3149–3235 completo? |
| :--- | ---: | ---: | :--- |
| 💥 `CHUNK_21` (corte fijo 150/0) | 3000 | 3150 | ❌ corta 85 caracteres antes de llegar a la excepción |
| 💥 `CHUNK_22` (corte fijo 150/0) | 3150 | 3300 | ❌ empieza en el carácter 3150 — un solo carácter después de donde arranca la regla (3149): la perdió por muy poco |
| ✅ `CHUNK_13` (recursivo 500/100) | 3119 | 3448 | ✅ arranca antes del 3149 y termina después del 3235: cubre las dos |

Con esos dos números el laboratorio puede hacerle al índice una pregunta que se responde sola —la misma que resuelve la tabla de arriba—: **¿existe algún fragmento cuyo `ini` sea menor o igual al inicio de la regla y cuyo `fin` sea mayor o igual al final de la excepción?**

Eso es lo que convierte una impresión —*"el chunking se ve mejor"*— en una métrica objetiva: la **integridad normativa** que vas a ver pasar de `0/4` a `4/4` en el Experimento 1. Sin los offsets habría que leer los fragmentos a ojo y decidir por intuición; con ellos, el veredicto lo emite el código.

In [ ]:
#@title 0.3 Motor RAG del laboratorio: corpus, chunker, indices y verificadores automaticos { display-mode: "form" }
import time, re, math, textwrap
import numpy as np
from collections import Counter
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================================================
# 1. BASE DOCUMENTAL CORPORATIVA (14 articulos con distractores deliberados)
#    El corpus esta disenado como un campo de pruebas: contiene codigos alfanumericos
#    casi identicos entre si, cuatro cifras de franquicia distintas y cuatro reglas
#    cuya EXCEPCION vive en el parrafo siguiente. Esos distractores son los que hacen
#    que un RAG mal calibrado falle de forma visible.
# =====================================================================================
DOCUMENTO_CORPORATIVO = """MANUAL DE POLIZAS Y PROCEDIMIENTOS OPERATIVOS 2024
Aseguradora Vertex Horizon Seguros S.A. - Gerencia de Suscripcion y Siniestros

ARTICULO 1: COBERTURA DE DANOS CLIMATICOS EN VIVIENDA URBANA
La poliza Hogar Total 2024 comprende la cobertura frente a contingencias meteorologicas como granizo, tempestades severas y anegamiento ordinario en cascos urbanos. La denuncia del evento climatico se canaliza mediante el formulario digital F-22 disponible en la sucursal virtual. Verificada la cobertura, la compania liquidara el 100% del peritaje evaluado con una franquicia fija deducible de 150 USD por evento.
EXCLUSION EXPRESA CLUST-90: Quedan excluidos de toda indemnizacion los siniestros derivados de inundaciones extraordinarias o desbordes de cauces hidricos en predios y zonas rurales que no cuenten con obras de mitigacion y diques de contencion homologados ante la Superintendencia de Seguros en el informe ambiental previo.

ARTICULO 2: PROTOCOLO DE DENUNCIA DE SINIESTROS VEHICULARES Y PLAZOS
Todo siniestro vehicular, sean colisiones simples, destruccion parcial o incidentes en autopista, debe ser denunciado ante el centro de atencion al asegurado dentro de un plazo perentorio improrrogable de setenta y dos horas habiles posteriores a la ocurrencia del hecho. El titular esta obligado a acompanar acta policial certificada, copia del registro de conducir vigente del conductor involucrado y el formulario mecanico de declaracion jurada. La omision del plazo de 72 horas faculta el rechazo administrativo por caducidad del derecho, conforme la clausula operativa RCT-19.

ARTICULO 3: RESCISION CONTRACTUAL ANTICIPADA Y COMPENSACION ECONOMICA
Ante la solicitud voluntaria de cancelacion del contrato formulada por el asegurado con anterioridad al vencimiento ordinario de la vigencia pactada, la compania restituira las primas netas correspondientes a los periodos no devengados. Sobre el saldo resultante se descuenta un arancel administrativo no reembolsable del 8% en concepto de gastos de emision.
EXCEPCION RES-22: Si la rescision se formaliza dentro de los primeros treinta dias corridos desde el inicio de vigencia, la restitucion sera integra y equivalente al 100% de las primas abonadas, sin aplicacion de arancel administrativo alguno.

ARTICULO 4: EXCLUSIONES ABSOLUTAS Y RESPONSABILIDAD DEL CONDUCTOR
Bajo ninguna circunstancia la aseguradora asumira cobertura, defensa judicial ni liquidacion de danos si se verifica que el conductor operaba el vehiculo bajo efectos comprobados de estupefacientes o con un dosaje de alcohol en sangre superior al limite legal fijado por la Ley Nacional de Transito, establecido en 0.5 g/l para conductores particulares. Asimismo, el dolo, la culpa grave del titular o la adulteracion de las circunstancias del siniestro extinguen toda obligacion resarcitoria de la entidad.

ARTICULO 5: COBERTURA DE ROBO TOTAL Y HURTO DE UNIDADES VEHICULARES
En los supuestos de desapoderamiento total de la unidad asegurada, la compania indemnizara el valor de reposicion vigente en la tabla de referencia del mercado automotor publicada por la autoridad competente. Sobre el valor de origen se aplica una depreciacion del 25% anual acumulada segun la antiguedad del rodado.
EXCLUSION EXC-114: Se deniega la cobertura de robo total cuando la unidad no contaba con el dispositivo de rastreo satelital homologado por la compania en estado operativo y activo al momento de la ocurrencia del hecho denunciado.

ARTICULO 6: COBERTURA DE GRANIZO SOBRE CULTIVOS Y PRODUCCION AGRICOLA
La poliza Agro Protegido 2024 ampara las perdidas de rendimiento ocasionadas por granizo sobre cultivos extensivos implantados. El resarcimiento reconoce un tope maximo indemnizable de 40 hectareas por explotacion y campana agricola declarada.
ANEXO ANX-7B: La cobertura agricola exige la declaracion fehaciente de siembra presentada con una antelacion minima de quince dias corridos respecto de la fecha de emergencia del cultivo, bajo pena de reduccion proporcional del resarcimiento.

ARTICULO 7: RESPONSABILIDAD CIVIL HACIA TERCEROS TRANSPORTADOS Y NO TRANSPORTADOS
La cobertura de responsabilidad civil ampara los danos materiales y lesiones causadas a terceros, comprendiendo la defensa penal y civil en sede judicial del asegurado, hasta un limite maximo unico y consolidado de 30000 USD por acontecimiento.

ARTICULO 8: COBERTURA DE INCENDIO, EXPLOSION Y RAYO EN INMUEBLES
Los danos materiales directos derivados de incendio, explosion accidental o impacto de rayo sobre el inmueble asegurado seran liquidados conforme al valor de reconstruccion a nuevo, aplicando una franquicia deducible de 300 USD por evento denunciado, segun el protocolo INC-57.

ARTICULO 9: SERVICIO DE ASISTENCIA MECANICA Y REMOLQUE EN VIA PUBLICA
El asegurado dispone de un maximo de seis servicios anuales de auxilio mecanico, con un radio de remolque cubierto de hasta 120 kilometros desde el punto de inmovilizacion del rodado hacia el taller designado.

ARTICULO 10: COBERTURA ESPECIFICA DE CRISTALES Y GRANIZO VEHICULAR
La rotura de parabrisas, lunetas y cristales laterales, asi como los danos de granizo sobre la carroceria del vehiculo asegurado, se liquidan con una franquicia deducible reducida de 90 USD por evento, sin afectar la bonificacion por siniestralidad del titular.

ARTICULO 11: RENOVACION AUTOMATICA Y ACTUALIZACION DE SUMAS ASEGURADAS
Los contratos se renuevan automaticamente al vencimiento, aplicandose un recargo de actualizacion tecnica del 12% sobre la prima vigente, salvo manifestacion expresa en contrario del asegurado con una antelacion de diez dias habiles.

ARTICULO 12: PAGO DE PRIMAS, MORA Y SUSPENSION DE COBERTURA
El asegurado dispone de un plazo de gracia de diez dias corridos desde el vencimiento de cada cuota para regularizar el pago. Vencido dicho plazo opera la suspension automatica de la cobertura conforme la disposicion DIC-45, reanudandose la vigencia unicamente desde la acreditacion efectiva del pago adeudado.

ARTICULO 13: AMBITO DE VIGENCIA TERRITORIAL Y PAISES LIMITROFES
La cobertura mantiene plena vigencia dentro del territorio nacional y en los estados parte del Mercosur, conforme el certificado de extension territorial MER-08 que debe ser solicitado con antelacion a cada viaje internacional del asegurado.

ARTICULO 14: PROCEDIMIENTO DE PERITAJE Y PLAZOS DE LIQUIDACION
Recibida la denuncia completa con la documentacion respaldatoria, la compania dispone de un plazo maximo de quince dias habiles para expedirse mediante dictamen pericial fundado, conforme el instructivo operativo PER-31."""

# Cuatro reglas cuya EXCEPCION vive en el parrafo siguiente (separadas por 69 a 97 caracteres).
# Si un chunk contiene la regla SIN su excepcion, el sistema promete pagar algo que el manual excluye.
PARES_CRITICOS = [
    {"nombre": "Art.1  Climatico  / CLUST-90", "regla": "liquidara el 100% del peritaje evaluado",  "excepcion": "EXCLUSION EXPRESA CLUST-90"},
    {"nombre": "Art.3  Rescision  / RES-22",   "regla": "arancel administrativo no reembolsable del 8%", "excepcion": "EXCEPCION RES-22"},
    {"nombre": "Art.5  Robo total / EXC-114",  "regla": "depreciacion del 25% anual acumulada",     "excepcion": "EXCLUSION EXC-114"},
    {"nombre": "Art.6  Agro       / ANX-7B",   "regla": "tope maximo indemnizable de 40 hectareas", "excepcion": "ANEXO ANX-7B"},
]

# =====================================================================================
# 2. SEGMENTADOR CON TRAZABILIDAD DE OFFSETS
#    Cada chunk recuerda que tramo exacto del documento original ocupa (ini, fin).
#    Eso permite medir objetivamente si una regla y su excepcion quedaron juntas.
# =====================================================================================
def _split_recursivo(texto, chunk_size, seps):
    if len(texto) <= chunk_size or not seps:
        return [texto]
    sep = seps[0]
    piezas, buf = [], ""
    for p in texto.split(sep):
        cand = (buf + sep + p) if buf else p
        if len(cand) <= chunk_size:
            buf = cand
        else:
            if buf:
                piezas.append(buf)
            if len(p) > chunk_size:
                piezas.extend(_split_recursivo(p, chunk_size, seps[1:])); buf = ""
            else:
                buf = p
    if buf:
        piezas.append(buf)
    return piezas

def segmentar_texto(texto, chunk_size=500, chunk_overlap=100, modo="recursivo"):
    """modo='fijo': corta cada N caracteres sin mirar el contenido.
       modo='recursivo': respeta parrafos > lineas > oraciones > palabras."""
    brutos = []
    if modo == "fijo":
        paso = max(1, chunk_size - chunk_overlap)
        for i in range(0, len(texto), paso):
            seg = texto[i:i + chunk_size]
            if seg.strip():
                brutos.append({"texto": seg.strip(), "ini": i, "fin": min(i + chunk_size, len(texto))})
            if i + chunk_size >= len(texto):
                break
    else:
        cursor, prev_fin = 0, None
        for p in _split_recursivo(texto, chunk_size, ["\n\n", "\n", ". ", " "]):
            p = p.strip()
            if not p:
                continue
            pos = texto.find(p, cursor)
            if pos < 0:
                pos = texto.find(p)
            ini, fin = pos, pos + len(p)
            cursor, t = fin, p
            if chunk_overlap > 0 and prev_fin is not None:
                ini_ov = max(0, ini - chunk_overlap)
                if ini_ov > 0:                                  # el solape no arranca a mitad de palabra
                    salto = texto.find(" ", ini_ov)
                    if 0 <= salto < ini:
                        ini_ov = salto + 1
                cola = texto[ini_ov:ini].strip()
                if cola:
                    t, ini = cola + " " + p, ini_ov
            brutos.append({"texto": t.strip(), "ini": ini, "fin": fin})
            prev_fin = fin

    chunks, ultimo = [], None
    for i, b in enumerate(brutos, 1):
        m = re.search(r"ARTICULO\s+(\d+)", b["texto"])
        art = f"ART-{int(m.group(1)):02d}" if m else None
        heredado = art is None
        if heredado:
            art = ultimo
        else:
            ultimo = art
        chunks.append({"id": f"CHUNK_{i:02d}", "texto": b["texto"], "longitud": len(b["texto"]),
                       "ini": b["ini"], "fin": b["fin"], "articulo": art, "heredado": heredado})
    return chunks

# =====================================================================================
# 3. METRICAS OBJETIVAS DE CALIDAD DEL INDICE (no dependen de leer los textos a ojo)
# =====================================================================================
def integridad_normativa(chunks, doc=DOCUMENTO_CORPORATIVO, pares=PARES_CRITICOS):
    """Cuenta cuantas reglas conservan su excepcion dentro de un MISMO chunk."""
    filas = []
    for p in pares:
        ir = doc.find(p["regla"])
        ie = doc.find(p["excepcion"]) + len(p["excepcion"])
        ok = any(c["ini"] <= ir and c["fin"] >= ie for c in chunks)
        filas.append({"par": p["nombre"], "intacto": ok})
    return filas

def pct_cortes_mutilados(chunks, doc=DOCUMENTO_CORPORATIVO):
    """% de chunks cuyo borde cae DENTRO de una palabra (sintoma del corte ciego)."""
    SEP = " \n\t.,;:()"
    def ini_ok(i): return i <= 0 or doc[i - 1] in SEP
    def fin_ok(i): return i >= len(doc) or doc[i] in SEP
    mal = sum(1 for c in chunks if not ini_ok(c["ini"]) or not fin_ok(c["fin"]))
    return 100.0 * mal / max(1, len(chunks))

def radiografia_indice(chunks, doc=DOCUMENTO_CORPORATIVO):
    integ = integridad_normativa(chunks, doc)
    return {"n_chunks": len(chunks),
            "pares_intactos": sum(1 for f in integ if f["intacto"]),
            "pares_total": len(integ),
            "pct_mutilados": pct_cortes_mutilados(chunks, doc),
            "largo_medio": sum(c["longitud"] for c in chunks) / max(1, len(chunks)),
            "detalle": integ}

# =====================================================================================
# 4. VISOR DEL MANUAL (plegado: se abre sólo si querés leer el documento completo)
# =====================================================================================
def ver_manual(resaltar_pares=True):
    """Muestra el manual de polizas completo, plegado. Con resaltar_pares=True marca
       las cuatro reglas criticas y sus excepciones para poder ubicarlas de un vistazo."""
    texto = DOCUMENTO_CORPORATIVO
    if resaltar_pares:
        for p in PARES_CRITICOS:
            texto = texto.replace(p["regla"], f"«REGLA» {p['regla']}")
            texto = texto.replace(p["excepcion"], f"«EXCEPCION» {p['excepcion']}")
    articulos = DOCUMENTO_CORPORATIVO.count("ARTICULO ")
    display(Markdown(
        f"<details>\n<summary><b>📄 Ver el Manual de Pólizas completo</b> "
        f"({len(DOCUMENTO_CORPORATIVO)} caracteres · {articulos} artículos · "
        f"{len(PARES_CRITICOS)} pares regla/excepción marcados)</summary>\n\n"
        f"```text\n{texto}\n```\n</details>"))

ver_manual()

### 🗂️ Qué es "indexar" (y por qué se hace antes de la primera pregunta)

La celda que sigue construye el **índice**. Es una palabra que va a aparecer en todo el laboratorio —"indexar el corpus", "reindexar", "la integridad del índice"— así que conviene fijar qué es exactamente.

Un índice es una estructura que se calcula **una sola vez, antes de que llegue la primera consulta**, para no tener que releer el documento entero en cada pregunta. Es análogo al índice alfabético del final de un libro: se arma antes y por eso la búsqueda es instantánea.

**Este laboratorio construye dos índices sobre exactamente los mismos fragmentos**, porque en el Experimento 2 vamos a enfrentarlos:

| Índice | Qué guarda de cada fragmento | Qué permite buscar |
| :--- | :--- | :--- |
| 🧠 **Denso** (embeddings) | un vector de 384 números que representa su *significado* | por idea: encuentra *"choque en autopista"* preguntando *"colisión vehicular"* |
| 🔤 **Léxico** (BM25) | qué palabras aparecen, cuántas veces y cuán raras son en el corpus | por palabra exacta: encuentra `EXC-114` porque dice `EXC-114` |

El orden importa, y es lo que explica dos advertencias que vas a leer más adelante:

`documento → fragmentos → índice → búsqueda`

Todo lo que pasa **antes de** la `búsqueda` ocurre **offline**, semanas antes de la consulta del cliente, y ya no se puede corregir en un sistema en *producción*. Si el corte partió una cláusula, ninguna pregunta bien formulada la va a recuperar. Y si se cambia el modelo de embeddings, hay que **reindexar**: los 384 números de cada fragmento se recalculan con otra regla y los vectores viejos dejan de ser comparables.

> 🎯 **La idea que hay que llevarse al Experimento 1:** el índice es todo lo que el buscador puede ver. Una cláusula que quedó mal indexada no está "difícil de encontrar" — para el sistema, directamente **no existe**.

In [ ]:
#@title 0.4 Motores de recuperación, verificadores automáticos y telemetría { display-mode: "form" }

# =====================================================================================
# 4. TRES MOTORES DE RECUPERACION: DENSO (embeddings), DISPERSO (BM25) E HIBRIDO (RRF)
# =====================================================================================
class BM25:
    """Busqueda dispersa clasica. Baseline honesto: usa IDF y normalizacion por longitud,
       no un simple conteo de palabras. Si aun asi falla, la falla es real."""
    STOP = set("""a al algo alguna algunas alguno algunos ante antes aquel aquella como con contra
    cual cuales cuando de del desde donde dos el ella ellas ellos en entre era eran es esa esas ese
    eso esos esta estan estas este esto estos ha han hasta hay la las le les lo los mas me mi mis
    mucho muy no nos o os otra otras otro otros para pero poco por porque que quien quienes se ser
    si sin sobre son su sus tan tanto te tengo tiene tienen toda todas todo todos tu tus un una uno
    unos y ya""".split())

    @staticmethod
    def tokenizar(t):
        t = t.lower()
        for a, b in zip("áéíóúñü", "aeiounu"):
            t = t.replace(a, b)
        return [w for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", t)
                if w not in BM25.STOP and len(w) > 2]

    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks, self.k1, self.b = chunks, k1, b
        self.docs = [BM25.tokenizar(c["texto"]) for c in chunks]
        self.tf = [Counter(d) for d in self.docs]
        self.N = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / max(1, self.N)
        self.df = Counter()
        for d in self.docs:
            for w in set(d):
                self.df[w] += 1

    def buscar(self, query, top_k=3):
        q = BM25.tokenizar(query)
        out = []
        for i, d in enumerate(self.docs):
            s = 0.0
            for w in q:
                f = self.tf[i].get(w, 0)
                if not f:
                    continue
                idf = math.log(1 + (self.N - self.df[w] + 0.5) / (self.df[w] + 0.5))
                s += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * len(d) / self.avgdl))
            if s > 0:
                out.append({"chunk": self.chunks[i], "score": s})
        return sorted(out, key=lambda r: r["score"], reverse=True)[:top_k]


class IndiceRAG:
    """Indexa una vez y expone los tres motores sobre exactamente los mismos chunks."""
    def __init__(self, chunks):
        self.chunks = chunks
        textos = [c["texto"] for c in chunks]
        self.embeddings = embedding_model.encode(textos, normalize_embeddings=True) if textos else None
        self.bm25 = BM25(chunks)

    def denso(self, query, top_k=3, umbral_minimo=0.0):
        if self.embeddings is None:
            return []
        q = embedding_model.encode([query], normalize_embeddings=True)
        sims = cosine_similarity(q, self.embeddings)[0]
        orden = np.argsort(sims)[::-1][:top_k]
        return [{"chunk": self.chunks[i], "score": float(sims[i])}
                for i in orden if float(sims[i]) >= umbral_minimo]

    def lexico(self, query, top_k=3):
        return self.bm25.buscar(query, top_k=top_k)

    def hibrido(self, query, top_k=3, k_rrf=60, pool=8):
        """Reciprocal Rank Fusion: fusiona los DOS rankings por posicion, no por score
           (los scores coseno y BM25 no son comparables entre si)."""
        d = self.denso(query, top_k=pool)
        l = self.lexico(query, top_k=pool)
        puntos = {}
        for ranking in (d, l):
            for pos, r in enumerate(ranking, start=1):
                cid = r["chunk"]["id"]
                puntos.setdefault(cid, {"chunk": r["chunk"], "score": 0.0})
                puntos[cid]["score"] += 1.0 / (k_rrf + pos)
        return sorted(puntos.values(), key=lambda r: r["score"], reverse=True)[:top_k]

    def buscar(self, query, motor="denso", top_k=3, umbral_minimo=0.0):
        if motor == "denso":
            return self.denso(query, top_k, umbral_minimo)
        if motor == "lexico":
            return self.lexico(query, top_k)
        return self.hibrido(query, top_k)


# =====================================================================================
# 5. VERIFICADORES AUTOMATICOS (mini-RAGAS): el veredicto lo emite el codigo, no el ojo
# =====================================================================================
def _norm_num(s):
    s = s.strip()
    if re.fullmatch(r"\d{1,3}([.,]\d{3})+", s):      # 30.000 -> 30000
        return s.replace(".", "").replace(",", "")
    return s.replace(",", ".")

def extraer_cifras(texto):
    """Cifras afirmadas en un texto, normalizadas (ignora anios y codigos tipo F-22)."""
    t = re.sub(r"(?m)^\s*\d{1,2}[.)]\s+", " ", texto)            # descarta numerales de lista (1. 2. 3.)
    t = re.sub(r"CHUNK_\d+", " ", t, flags=re.I)                 # descarta ids de fragmento
    t = re.sub(r"\b[A-Z]{2,6}-\d+[A-Z]?\b", " ", t)              # descarta codigos de clausula
    crudas = re.findall(r"\d{1,3}(?:[.,]\d{3})+|\d+(?:[.,]\d+)?", t)
    return {_norm_num(c) for c in crudas if _norm_num(c) not in {"2024", "2025"}}

FRASE_ABSTENCION = "no poseo informacion oficial"

# El modelo escribe indistintamente "72 horas" o "setenta y dos horas": para la ficha de
# evaluacion ambas formas cuentan como haber citado el dato correcto.
NUM_EN_PALABRAS = {
    "6": ["seis"], "8": ["ocho"], "10": ["diez"], "12": ["doce"], "15": ["quince"],
    "20": ["veinte"], "25": ["veinticinco", "veinte y cinco"], "30": ["treinta"],
    "40": ["cuarenta"], "72": ["setenta y dos"], "90": ["noventa"],
    "120": ["ciento veinte"], "150": ["ciento cincuenta"], "300": ["trescientos"],
    "30000": ["treinta mil"],
}

def auditar_respuesta(respuesta, contexto, ficha):
    """Devuelve el diagnostico objetivo de una respuesta contra su ficha de evaluacion."""
    r_norm = respuesta.lower()
    for a, b in zip("áéíóú", "aeiou"):
        r_norm = r_norm.replace(a, b)

    cifras_resp = extraer_cifras(respuesta)
    cifras_ctx = extraer_cifras(contexto)
    inventadas = sorted(cifras_resp - cifras_ctx, key=lambda x: float(x))
    prohibidas = sorted(cifras_resp & set(ficha.get("cifras_prohibidas", [])), key=lambda x: float(x))
    esperadas = set(ficha.get("cifras_esperadas", []))
    cubiertas = sorted([c for c in esperadas
                        if c in cifras_resp or any(p in r_norm for p in NUM_EN_PALABRAS.get(c, []))],
                       key=lambda x: float(x))

    ids_ctx = set(re.findall(r"CHUNK_\d+", contexto.upper()))
    citadas = set(re.findall(r"CHUNK_\d+", respuesta.upper()))
    citas_ok, citas_falsas = sorted(citadas & ids_ctx), sorted(citadas - ids_ctx)

    faltantes = [t for t in ficha.get("terminos_obligatorios", []) if t.lower() not in r_norm]
    se_abstuvo = FRASE_ABSTENCION in r_norm
    debe_abstenerse = ficha.get("debe_abstenerse", False)

    if debe_abstenerse:
        exito = se_abstuvo and not prohibidas and not citas_falsas
    else:
        exito = ((not inventadas) and (not prohibidas) and bool(cubiertas)
                 and not se_abstuvo and not citas_falsas and not faltantes)

    return {"cifras_inventadas": inventadas, "cifras_prohibidas": prohibidas,
            "cifras_correctas": cubiertas, "citas_validas": citas_ok,
            "citas_falsas": citas_falsas, "se_abstuvo": se_abstuvo,
            "debe_abstenerse": debe_abstenerse, "exito": exito,
            "trazable": bool(citas_ok) and not citas_falsas,
            "terminos_faltantes": faltantes}

def semaforo(auditoria):
    a = auditoria
    if a["exito"]:
        return "🟢 CORRECTA"
    if a["citas_falsas"]:
        return "🔴 CITA INEXISTENTE"
    if a["cifras_prohibidas"]:
        return "🔴 ALUCINACION POR CONTAMINACION"
    if a["cifras_inventadas"]:
        return "🔴 ALUCINACION NUMERICA"
    if a.get("terminos_faltantes"):
        return "🔴 OMITE LA EXCLUSION"
    if a["debe_abstenerse"] and not a["se_abstuvo"]:
        return "🔴 NO SE ABSTUVO"
    if a["se_abstuvo"] and not a["debe_abstenerse"]:
        return "🟠 ABSTENCION INDEBIDA"
    return "🟠 INCOMPLETA / SIN DATO"


# =====================================================================================
# 6. INFERENCIA CON TELEMETRIA REAL (TTFT / TPOT / EERL, tal como se definieron en teoria)
# =====================================================================================
# El modelo corre en VRAM local: no hay factura por token, asi que dejamos que termine sus
# respuestas en lugar de cortarlas. El tope existe solo como red de seguridad ante el bucle de
# repeticion tipico de los modelos pequenos cuantizados, que sin limite colgaria la celda.
MAX_TOKENS_RESPUESTA = 1024

def ejecutar_consulta(prompt, system_prompt="Eres un asistente analitico riguroso y conciso.",
                      temperature=0.1, top_p=0.9, max_tokens=None, repeat_penalty=1.15,
                      contexto_referencia="", mostrar=True):
    # repeat_penalty > 1 penaliza repetir tokens ya emitidos. Sin esto, un modelo de 3B sin
    # contexto puede entrar en bucle ("...de coche de coche de coche...") y agotar el presupuesto.
    max_tokens = MAX_TOKENS_RESPUESTA if max_tokens is None else max_tokens
    # Compatibilidad con modelos que no admiten el rol "system" (familia Gemma):
    # se funde el system prompt dentro del mensaje de usuario. En este laboratorio
    # importa especialmente, porque en el system prompt viven la regla de abstencion
    # y la cita obligatoria del Experimento 3.
    es_gemma = "gemma" in llm.model_path.lower()
    if es_gemma:
        mensajes = [{"role": "user",
                     "content": f"Instrucciones del Sistema: {system_prompt}\n\nTarea/Pregunta: {prompt}"}]
    else:
        mensajes = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt}]
    t0 = time.time()
    ttft, piezas, n_out = None, [], 0
    try:
        for ev in llm.create_chat_completion(messages=mensajes, temperature=temperature,
                                             top_p=top_p, max_tokens=max_tokens,
                                             repeat_penalty=repeat_penalty, stream=True):
            delta = ev["choices"][0].get("delta", {}).get("content")
            if delta:
                if ttft is None:
                    ttft = time.time() - t0
                piezas.append(delta); n_out += 1
    except Exception as err:
        display(Markdown(f"⚠️ **Falla durante la inferencia:** `{err}`"))
        return ""
    total = time.time() - t0
    texto = "".join(piezas).strip()
    ttft = ttft if ttft is not None else total
    try:
        n_in = len(llm.tokenize((system_prompt + prompt).encode("utf-8")))
    except Exception:
        n_in = -1
    tpot = (total - ttft) / max(1, n_out - 1)

    truncada = n_out >= max_tokens
    if mostrar:
        # Tres bloques SEPARADOS, en este orden y siempre con una linea en blanco entre medio:
        # 1) la respuesta  2) el contexto plegado  3) la telemetria como tabla.
        # La tabla mantiene los numeros alineados y comparables entre generaciones, que es lo
        # que pide el objetivo 6 del laboratorio (leer TTFT/TPOT/EERL como insumo de decision).
        bloques = [f"### 💬 Salida del modelo\n\n{texto}"]
        if truncada:
            bloques.append("> ⚠️ **Respuesta cortada por el tope de tokens** — probablemente el "
                           "modelo entró en un bucle de repetición.")
        if contexto_referencia:
            bloques.append("<details>\n<summary><b>👁️ Inspeccionar el contexto exacto que recibió "
                           f"el modelo</b></summary>\n\n```text\n{contexto_referencia}\n```\n"
                           "</details>")
        pie = f"{llm.model_path.split('/')[-1]} · Temp {temperature} · Top-p {top_p} · Repeat-penalty {repeat_penalty}"
        if es_gemma:
            pie += " · ⚠️ sin rol system: las instrucciones van fundidas en el mensaje de usuario"
        bloques.append(
            "| ⏱️ TTFT | 🏁 EERL | ⚡ TPOT | 🧮 Tokens |\n"
            "| ---: | ---: | ---: | ---: |\n"
            f"| {ttft:.2f} s | {total:.2f} s | {tpot*1000:.0f} ms/tok ({1/tpot if tpot>0 else 0:.1f} tok/s) | "
            f"{n_in} → {n_out} |\n\n"
            f"<sub>{pie}</sub>")
        display(Markdown("\n\n---\n\n".join(bloques)))
    return {"texto": texto, "ttft": ttft, "eerl": total, "tok_in": n_in, "tok_out": n_out,
            "truncada": truncada, "max_tokens": max_tokens}


# =====================================================================================
# 7. INDICE POR DEFECTO (configuracion sana; los experimentos la reindexan a voluntad)
# =====================================================================================
CHUNKS_SANOS = segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=500, chunk_overlap=100, modo="recursivo")
indice = IndiceRAG(CHUNKS_SANOS)

_r = radiografia_indice(CHUNKS_SANOS)
print(f"✅ Corpus: {len(DOCUMENTO_CORPORATIVO)} caracteres · 14 articulos · 9 codigos de clausula")
print(f"✅ Indice por defecto: {_r['n_chunks']} chunks · integridad normativa "
      f"{_r['pares_intactos']}/{_r['pares_total']} · cortes mutilados {_r['pct_mutilados']:.0f}%")
print("✅ Motores listos: denso (embeddings) · disperso (BM25) · hibrido (RRF)")
print("✅ Verificadores automaticos y telemetria TTFT/TPOT/EERL operativos.")

> 🩺 **Antes de empezar.** Corré la celda `0.5` una sola vez, apenas terminen de montarse los modelos de la celda `0.2`. Tarda segundos, no usa el modelo generador, y deja a la vista la línea base de los cuatro experimentos medida sobre tus propios datos. Es a lo que vas a querer volver cuando muevas una perilla y el resultado te sorprenda.

In [ ]:
#@title 🩺 0.5 — ¿Quedó todo bien montado? { display-mode: "form" }
# Cuatro comprobaciones rapidas sobre los datos de ESTA sesion. No llama al modelo generador:
# solo usa el motor de embeddings, asi que tarda segundos y no gasta GPU de generacion.
#
# Para que sirve: los cuatro experimentos arrancan de un punto de referencia conocido. Cuando mas
# adelante muevas una perilla y el resultado te sorprenda, volver aca te dice en cinco segundos si
# lo que estas viendo es un hallazgo tuyo o si algo del montaje quedo distinto.
#
# Nota de diseno: usa copias MINIMAS y propias de las consultas de cada experimento, para poder
# ejecutarse sin depender de haber corrido esas celdas. Si editas las consultas originales de un
# experimento, actualiza tambien su copia de aca abajo.

_cal = []   # (etiqueta, que mide, valor medido, ok, lectura si NO esta ok)

# ------------------------------------------------- EXPERIMENTO 1 · segmentación (determinista)
_c_rot = radiografia_indice(segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=150, chunk_overlap=0, modo="fijo"))
_c_med = radiografia_indice(segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=500, chunk_overlap=0, modo="recursivo"))
_c_san = radiografia_indice(segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=500, chunk_overlap=100, modo="recursivo"))
_cal.append((
    "✂️ **Experimento 1 · segmentación**",
    "reglas que conservan su excepción, en las tres configuraciones",
    f"{_c_rot['pares_intactos']}/4 → {_c_med['pares_intactos']}/4 → {_c_san['pares_intactos']}/4",
    _c_rot["pares_intactos"] == 0 and _c_san["pares_intactos"] == 4 and _c_rot["pct_mutilados"] > 80,
    "Esta comprobación es puramente aritmética: si falla, el corpus del notebook fue editado."))

# ------------------------------------------------- EXPERIMENTO 2 · recuperación (depende del encoder)
_CAL_CONSULTAS_E2 = [
    ("Q1", "jerga coloquial", "Si doy de baja todo antes de que se termine, ¿me devuelven algo de lo que puse?", "ART-03"),
    ("Q2", "codigo exacto",   "¿Qué establece exactamente la cláusula EXC-114?", "ART-05"),
    ("Q3", "distractor",      "¿Qué franquicia se aplica si el granizo me rompe el parabrisas del auto?", "ART-10"),
    ("Q4", "fuera de dominio","¿Cuáles son los requisitos para un crédito hipotecario de vivienda?", None),
]
_c_marcador = {"denso": 0, "lexico": 0, "hibrido": 0}
for _qid, _tipo, _texto, _gt in _CAL_CONSULTAS_E2:
    for _motor in ("denso", "lexico", "hibrido"):
        _rs = indice.buscar(_texto, motor=_motor, top_k=2)
        _arts = [r["chunk"]["articulo"] for r in _rs]
        _c_marcador[_motor] += int((len(_rs) == 0) if _gt is None else (_gt in _arts))
_c_bm25_mudo = len(indice.lexico(_CAL_CONSULTAS_E2[0][2], top_k=2)) == 0
_cal.append((
    "🔍 **Experimento 2 · recuperación**",
    "aciertos sobre 4 consultas con `Top-K = 2`",
    f"denso {_c_marcador['denso']} · BM25 {_c_marcador['lexico']} · híbrido {_c_marcador['hibrido']}",
    _c_marcador["hibrido"] >= max(_c_marcador["denso"], _c_marcador["lexico"]) and _c_bm25_mudo,
    "El híbrido debería igualar o superar a los dos motores puros, y BM25 debería quedarse "
    "mudo ante la jerga de Q1. Si no pasa, cambió el modelo de embeddings."))

# ------------------------------------------------- EXPERIMENTO 4 · umbral (depende del encoder)
_CAL_SET_EVAL_E4 = [
    ("¿Cuántas horas tengo para denunciar un choque en la autopista?",        True),
    ("Quiero cancelar la póliza antes de tiempo, ¿qué me descuentan?",        True),
    ("¿Hasta qué monto cubre la responsabilidad civil frente a terceros?",    True),
    ("¿Cuáles son los requisitos para un crédito hipotecario a 20 años?",     False),
    ("¿Cómo tramito el pasaporte y la visa de turista para Europa?",          False),
    ("¿Qué rendimiento anual paga hoy un plazo fijo en pesos?",               False),
]
_CAL_UMBRAL_E4 = 0.50   # debe coincidir con UMBRAL_E4 en la celda 4.A
_c_dentro = [indice.denso(q, top_k=1)[0]["score"] for q, c in _CAL_SET_EVAL_E4 if c]
_c_fuera  = [indice.denso(q, top_k=1)[0]["score"] for q, c in _CAL_SET_EVAL_E4 if not c]
_c_qa = indice.denso("¿Cuáles son los requisitos y la tasa de interés para un crédito hipotecario a 20 años?", top_k=1)[0]["score"]
_c_qb = indice.denso("¿Cuál es la franquicia deducible que debo pagar por el robo total de mi auto?", top_k=1)[0]["score"]
_c_sugerido = round((min(_c_dentro) + max(_c_fuera)) / 2, 2)
_cal.append((
    "🎯 **Experimento 4 · umbral**",
    f"peor consulta cubierta contra la mejor ajena, umbral `{_CAL_UMBRAL_E4:.2f}`",
    f"{min(_c_dentro):.2f} dentro · {max(_c_fuera):.2f} fuera",
    (_c_qa < _CAL_UMBRAL_E4) and (_c_qb >= _CAL_UMBRAL_E4),
    f"El umbral tiene que bloquear la pregunta de otro dominio y **dejar pasar** la del dato "
    f"inexistente, que es el punto ciego del Experimento 4. Con estos scores, el valor que las "
    f"separa es `{_c_sugerido:.2f}`."))

# ------------------------------------------------- EXPERIMENTO 3 · la trampa está armada
_c_ctx = "\n\n".join(r["chunk"]["texto"] for r in
                     indice.denso("¿Cuál es la franquicia deducible que debo pagar por el robo total de mi auto?", top_k=3))
_c_ajenas = sorted(extraer_cifras(_c_ctx) & {"150", "300", "90"})
_cal.append((
    "🧨 **Experimento 3 · la trampa**",
    "franquicias de *otras* coberturas visibles en el contexto de la pregunta trampa",
    ", ".join(f"`{c}`" for c in _c_ajenas) if _c_ajenas else "ninguna",
    bool(_c_ajenas),
    "Sin cifras ajenas a la vista, el RAG ingenuo no tiene de dónde copiar una franquicia "
    "equivocada y la trampa del Experimento 3 no se dispara."))

# ------------------------------------------------- Salida
_todo_ok = all(ok for _, _, _, ok, _ in _cal)
_txt = ("### 🩺 El punto de partida de los cuatro experimentos\n\n"
        "Cuatro mediciones sobre tus propios datos, sin llamar al modelo generador. "
        "Así se ve el sistema **antes** de que muevas nada.\n\n"
        "| Qué se comprueba | Lo que mide | Valor medido | Estado |\n"
        "| :--- | :--- | :--- | :---: |\n")
for _etq, _mide, _val, _ok, _ in _cal:
    _txt += f"| {_etq} | {_mide} | {_val} | {'🟢' if _ok else '🟠'} |\n"

if _todo_ok:
    _txt += ("\n**Todo montado como corresponde.** Estos cuatro números son tu línea base: cuando en "
             "un experimento muevas una perilla y el resultado te sorprenda, volvé acá para "
             "distinguir un hallazgo tuyo de un montaje distinto. Podés seguir a la celda `1.A`.\n")
else:
    _txt += "\n**Algo quedó distinto de la línea base:**\n\n"
    for _etq, _, _, _ok, _lectura in _cal:
        if not _ok:
            _txt += f"* {_etq} — {_lectura}\n"
    _txt += ("\nNo es un error del notebook. Los contrastes están calibrados con el modelo "
             "predeterminado de la celda `0.2`: si cambiaste de modelo, **esto ya es un hallazgo** "
             "y vale la pena anotarlo para el TP2.\n")
display(Markdown(_txt))

## 🔬 Experimento 1 — Segmentación: dónde se pierden las excepciones legales

### 🎯 El problema
La calidad de un sistema RAG está acotada por la calidad de su segmentación. Es la decisión más influyente de todo el pipeline y, además, la más traicionera: **un chunking defectuoso no produce ningún error visible**. No hay excepción, no hay log, no hay alerta. Simplemente, ciertas cláusulas dejan de existir para el buscador.

**Modelo mental — la tijera y el libro de recetas:** si cortás un recetario cada 200 caracteres, una tira de papel instruye con total claridad: «Desmoldar inmediatamente al retirar del horno». La tira siguiente advertía: «Salvo que se haya utilizado molde de silicona, en cuyo caso debe enfriarse por completo o la estructura colapsará». Quien cocine con el primer fragmento seguirá el paso al pie de la letra, pero arruinará el plato porque la excepción invalidante quedó fuera de su vista. En un manual de seguros el equivalente es exactamente el mismo: **la regla general y la excepción que la anula quedan en fragmentos distintos**, y el sistema entrega la promesa de pago sin la condición que la limita.

### 🧪 Qué vamos a medir
El manual tiene **cuatro reglas cuya excepción vive en el párrafo siguiente**, a una distancia de entre 69 y 97 caracteres. Sobre eso definimos dos métricas objetivas:

| Métrica | Qué cuenta | Qué parámetro la gobierna |
| :--- | :--- | :--- |
| 🧩 **Integridad normativa** | De las 4 reglas, cuántas conservan su excepción en el *mismo* chunk | el **solape** |
| ✂️ **Cortes mutilados** | % de fragmentos cuyo borde cae dentro de una palabra | la **estrategia** (fija vs. recursiva) |

### 📋 Consigna de exploración
1. **Ejecutá la celda `1.A`.** Las tres primeras filas son fijas —el ancla del experimento— y cada una arregla exactamente **una** dimensión: el corte rígido falla en ambas; el recursivo sin solape arregla los cortes pero **sigue perdiendo las cuatro excepciones**; el recursivo con solape arregla las dos. La **cuarta fila es tuya**: movés las perillas del formulario y comparás tu resultado contra esas tres anclas. Probá, por ejemplo, `fijo 300/50`: recupera los 4 pares y aun así parte el 96% de las palabras —ganás un eje y perdés el otro—.
2. **Ejecutá la celda `1.B`.** Ahí movemos una sola perilla —el solape— y observamos en qué valor exacto el índice pasa de `0/4` a `4/4`. Anotá ese número y compará con la recomendación teórica.
3. **Ejecutá la celda `1.C`.** Es el cierre: la misma pregunta de un cliente, contra el índice roto y contra el sano. Mismo modelo, mismo prompt, mismo Top-K. Sólo cambia cómo se cortó el documento.

In [ ]:
#@title 🚀 1.A — Comparación automática: las tres configuraciones, lado a lado { display-mode: "form", run: "auto" }
# Indexa el MISMO documento con tres estrategias y mide la calidad del índice resultante.
# No hay que interpretar textos a ojo: las métricas emiten el veredicto.
#
# Las tres columnas de CONFIGS_E1 son el ANCLA del experimento: se recalculan siempre, muevas lo
# que muevas en el formulario de abajo. El formulario agrega una CUARTA columna propia -nunca
# reemplaza las tres calibradas- para que el contraste se lea de izquierda a derecha sin perderse.

#@markdown Las tres configuraciones calibradas se calculan siempre. Este formulario agrega una **cuarta columna tuya**, para comparar:
estrategia_tuya = "recursivo"  #@param ["recursivo", "fijo"]
chunk_size_tuyo = 500          #@param {type:"slider", min:100, max:1200, step:50}
overlap_tuyo    = 100          #@param {type:"slider", min:0, max:300, step:25}

# La CLAVE es estable ("fijo"/"sin_solape"/"con_solape") y la etiqueta visible va aparte:
# asi cambiar la presentacion no rompe el estado que consume 1.C.
CONFIGS_E1 = [
    ("fijo",       "Fijo 150",                  dict(chunk_size=150, chunk_overlap=0,   modo="fijo")),
    ("sin_solape", "Recursivo 500",             dict(chunk_size=500, chunk_overlap=0,   modo="recursivo")),
    ("con_solape", "Recursivo 500+100",          dict(chunk_size=500, chunk_overlap=100, modo="recursivo")),
]

indices_e1 = {}
for clave, etiqueta, cfg in CONFIGS_E1:
    chs = segmentar_texto(DOCUMENTO_CORPORATIVO, **cfg)
    indices_e1[clave] = (chs, cfg, radiografia_indice(chs))

# --- Cuarta columna: TU configuración, en variables propias -----------------------------------
# NO se agrega a indices_e1: esa variable alimenta la celda 1.C, y si el estudiante la rebindeara
# el cierre causal del experimento quedaría midiendo una configuración distinta sin avisar.
cfg_tuya = dict(chunk_size=chunk_size_tuyo, chunk_overlap=overlap_tuyo, modo=estrategia_tuya)
chunks_tuyos = segmentar_texto(DOCUMENTO_CORPORATIVO, **cfg_tuya)
rad_tuya = radiografia_indice(chunks_tuyos)

def _indice_almacenado(chs):
    """Cuanto pesa el indice contra el corpus original: 1.00x = sin redundancia."""
    return sum(c["longitud"] for c in chs) / len(DOCUMENTO_CORPORATIVO)

# Las cuatro columnas de la tabla, en orden. Cada una: (encabezado, chunks, radiografia).
COLS = [(et, indices_e1[k][0], indices_e1[k][2]) for k, et, _ in CONFIGS_E1]
COLS.append(("👤 Tu config", chunks_tuyos, rad_tuya))

def _fila(titulo, celda_fn):
    return f"| {titulo} | " + " | ".join(celda_fn(chs, rad) for _, chs, rad in COLS) + " |\n"

# Todas las metricas apuntan en la misma direccion: mas alto = mejor. Por eso los cortes se
# informan como "limpios" (100 - mutilados) y no como "palabras partidas".
tabla = f"""### 🎯 Cómo se corta el manual decide qué cláusulas existen

Mismo documento, mismo modelo, misma pregunta. Lo único que cambia es dónde caen los cortes.

| | {" | ".join(t for t, _, _ in COLS)} |
| :--- |{" :---: |" * len(COLS)}
"""
tabla += _fila("Fragmentos", lambda chs, rad: f"{rad['n_chunks']}")
tabla += _fila("✂️ Cortes limpios", lambda chs, rad:
               f"{'✅' if rad['pct_mutilados'] < 5 else '❌'} {100 - rad['pct_mutilados']:.0f}%")
tabla += _fila("🧩 Reglas con su excepción", lambda chs, rad:
               f"{'✅' if rad['pares_intactos'] == rad['pares_total'] else '❌'} "
               f"{rad['pares_intactos']} de {rad['pares_total']}")
tabla += _fila("💾 Peso del índice", lambda chs, rad: f"{_indice_almacenado(chs):.2f}×")

tabla += """
**Cada columna arregla exactamente una dimensión más que la anterior.** El corte fijo falla en las dos; el recursivo endereza los cortes pero sigue perdiendo las cuatro excepciones; recién el solape las rescata.

* **✂️ Cortes limpios** — % de fragmentos que empiezan y terminan en un borde de palabra. Cortar por cantidad de caracteres parte las palabras al medio.
* **🧩 Reglas con su excepción** — de las 4 reglas del manual que tienen una excepción en el párrafo siguiente, cuántas conservan esa excepción dentro del *mismo* fragmento. Es la métrica que decide si el sistema puede prometer un pago sin la condición que lo limita.
* **💾 Peso del índice** — cuánto ocupa el índice contra el documento original (`1.00×` = sin redundancia). El solape repite texto a propósito, y eso se vuelve a vectorizar, a almacenar y a pagar en tokens.

---

#### 🔍 ¿Qué excepción sobrevivió en cada caso?

"""
tabla += f"| Regla y su excepción | {' | '.join(t for t, _, _ in COLS)} |\n"
tabla += f"| :--- |{' :---: |' * len(COLS)}\n"
for i, par in enumerate(PARES_CRITICOS):
    celdas = " | ".join("✅" if rad["detalle"][i]["intacto"] else "❌" for _, _, rad in COLS)
    tabla += f"| `{par['nombre']}` | {celdas} |\n"

display(Markdown(tabla))

# --- Veredicto comparativo: tu configuración contra las dos anclas extremas --------------------
rot = indices_e1["fijo"][2]
integ_tuya = f"{rad_tuya['pares_intactos']}/{rad_tuya['pares_total']}"
almac_tuya = _indice_almacenado(chunks_tuyos)
faltan_tuya = [p["par"] for p in rad_tuya["detalle"] if not p["intacto"]]

coincide = next((et for _, et, cfg in CONFIGS_E1
                 if cfg["modo"] == estrategia_tuya and cfg["chunk_size"] == chunk_size_tuyo
                 and cfg["chunk_overlap"] == overlap_tuyo), None)

if coincide:
    # Caso por defecto: la columna del estudiante duplica un ancla. En vez de compararla contra
    # si misma, se confirma la coincidencia y se invita a mover las perillas.
    veredicto = (f"👤 **Tu configuración** (`{estrategia_tuya} {chunk_size_tuyo}/{overlap_tuyo}`) es "
                 f"exactamente el ancla **{coincide}**: {integ_tuya} pares, "
                 f"{100 - rad_tuya['pct_mutilados']:.0f}% de cortes limpios, índice de {almac_tuya:.2f}×. "
                 f"Mové las perillas de arriba para comparar otra cosa contra las tres anclas.")
else:
    veredicto = (f"👤 **Tu configuración** (`{estrategia_tuya} {chunk_size_tuyo}/{overlap_tuyo}`) "
                 f"recupera **{integ_tuya}** pares")
    if rad_tuya["pares_intactos"] == 4:
        veredicto += " — **el índice queda completo**, igual que el ancla con solape."
    elif rad_tuya["pares_intactos"] > rot["pares_intactos"]:
        veredicto += (f", mejor que el corte fijo pero todavía por debajo del ancla con solape. "
                      f"Te faltan: {', '.join(faltan_tuya)}.")
    else:
        veredicto += f" — igual de mal que el corte fijo. Te faltan: {', '.join(faltan_tuya)}."

if rad_tuya["pares_intactos"] == 4 and rad_tuya["pct_mutilados"] >= 5:
    veredicto += (f"\n\n⚠️ Ganaste la integridad normativa, pero a costa de los cortes: sólo "
                  f"**{100 - rad_tuya['pct_mutilados']:.0f}% de los fragmentos** empieza y termina "
                  f"en un borde de palabra. Es el mismo síntoma que el corte fijo, sólo que "
                  f"disimulado por un solape grande.")

if almac_tuya >= 3.0:
    veredicto += (f"\n\n🚨 Tu índice almacena **{almac_tuya:.1f}×** el documento original: con un "
                  f"`overlap` tan cercano (o mayor) al `chunk_size`, dejaste de segmentar y "
                  f"empezaste a deslizar una ventana. Cada carácter repetido se vectoriza, se "
                  f"guarda y se vuelve a pagar en tokens cuando entra al prompt.")

display(Markdown(veredicto))

# Evidencia textual: el mismo tramo del manual, visto por el índice roto y por el sano.
# Lo ubicamos por OFFSET y no por texto: en el índice roto hasta las palabras están partidas.
# Regla de presentacion del notebook: el texto del fragmento va en bloque de codigo (es lo que
# recibe la maquina) y el comentario va afuera, como prosa. Nunca los dos como blockquote.
POS_REGLA = DOCUMENTO_CORPORATIVO.find("depreciacion del 25%")
POS_EXC   = DOCUMENTO_CORPORATIVO.find("EXCLUSION EXC-114")

def _chunk_en(chunks, pos):
    return next((c for c in chunks if c["ini"] <= pos < c["fin"]), None)

roto = _chunk_en(indices_e1["fijo"][0], POS_REGLA)
sano = _chunk_en(indices_e1["con_solape"][0], POS_EXC)

ev = ("#### 🧾 El Artículo 5 tal como lo ve el buscador\n\n"
      "El mismo tramo del manual —la regla de indemnización por robo total y su exclusión— "
      "leído desde dos índices distintos.\n\n")
if roto:
    ev += (f"**💥 Índice roto — `{roto['id']}`** · {roto['longitud']} caracteres\n\n"
           f"```text\n{' '.join(roto['texto'].split())}\n```\n\n"
           f"**No contiene `EXC-114`.** El fragmento arranca y termina a mitad de palabra, promete "
           f"indemnizar y no dice nada del rastreo satelital obligatorio. Un asegurado sin "
           f"rastreador recibiría una promesa de pago que el manual **deniega**.\n\n---\n\n")
if sano:
    ev += (f"**✅ Índice sano — `{sano['id']}`** · {sano['longitud']} caracteres\n\n"
           f"```text\n{' '.join(sano['texto'].split())}\n```\n\n"
           f"**Contiene la regla y la exclusión en el mismo fragmento.** Gracias al solape, el "
           f"generador no puede prometer el pago sin ver la condición que lo anula.\n")
display(Markdown(ev))


In [ ]:
#@title 🚀 1.B — Barrido gradual: ¿cuánto solape hace falta para no perder las excepciones? { display-mode: "form" }
# Mantiene fija la estrategia (recursiva, 500 caracteres) y mueve UNA sola perilla: el solape.
#
# Los seis valores no son arbitrarios: barriendo de 0 a 160 de 10 en 10, las UNICAS transiciones
# estan en 70 (0->1), 90 (1->3) y 100 (3->4). Cada fila de abajo tiene un motivo propio:
#   0   punto de partida            70  primera excepcion rescatada    100  la frontera
#   50  el piso del 10% que         90  la pendiente                   110  pasada la frontera
#       recomienda la teoria                                                no se gana nada mas
SIZE_FIJO = 500
barrido = []
for ov in [0, 50, 70, 90, 100, 110]:
    chs = segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=SIZE_FIJO, chunk_overlap=ov, modo="recursivo")
    rad = radiografia_indice(chs)
    sobreviven = [p["par"].split()[0] for p in rad["detalle"] if p["intacto"]]
    barrido.append((ov, rad, sobreviven))

txt = f"""### 🎯 Una sola perilla decide cuántas cláusulas sobreviven

Estrategia recursiva y `chunk_size = {SIZE_FIJO}` fijos en todas las filas. Lo único que se mueve es el solape.

| Overlap | % del chunk | Integridad normativa | Barra | Excepciones rescatadas |
| ---: | ---: | :---: | :--- | :--- |
"""
for ov, rad, sob in barrido:
    n = rad["pares_intactos"]
    barra = "█" * n + "░" * (rad["pares_total"] - n)
    txt += (f"| `{ov}` | {100*ov/SIZE_FIJO:.0f}% | **{n}/{rad['pares_total']}** | "
            f"`{barra}` | {', '.join(sob) if sob else '—'} |\n")

primero_pleno = next((ov for ov, rad, _ in barrido if rad["pares_intactos"] == rad["pares_total"]), None)
txt += f"""
> 📌 **Lectura del barrido:** las cuatro excepciones del manual están entre **69 y 97 caracteres** después de su regla. Mientras el solape no alcanza esa distancia, el índice pierde cláusulas de forma **silenciosa**: nada falla, nada avisa, simplemente la excepción deja de existir para el buscador. El índice queda completo recién con **overlap = {primero_pleno}** ({100*primero_pleno/SIZE_FIJO:.0f}% del chunk), que es el techo de la franja del **10–20%** que recomienda la teoría — fijate que con el **piso** de esa franja (overlap 50) este corpus todavía pierde las cuatro.
>
> ⚠️ Y el solape no es gratis: mirá la última fila. Pasada la frontera no se rescata ninguna excepción nueva, pero cada carácter repetido se vuelve a vectorizar, se vuelve a almacenar y se vuelve a pagar en la factura de tokens cuando entra al prompt.
"""
display(Markdown(txt))


In [ ]:
#@title 🚀 1.C — Cierre causal: la misma pregunta contra el índice roto y contra el índice sano { display-mode: "form" }
# Hasta acá medimos el índice. Ahora medimos la CONSECUENCIA: qué le contesta el sistema
# a un cliente real. Cambia una sola cosa entre las dos corridas: cómo se cortó el documento.

PREGUNTA_E1 = "Me robaron el auto que tengo asegurado. ¿Cómo calculan el monto que me van a pagar?"

FICHA_E1 = {
    "cifras_esperadas": ["25"],                       # depreciación del 25% anual (Art. 5)
    "cifras_prohibidas": [],
    "debe_abstenerse": False,
    "terminos_obligatorios": ["satelital"],           # EXC-114 condiciona TODO el pago
}

SYS_E1 = ("Actuas como liquidador de siniestros de Vertex Horizon Seguros. Responde en español, en un "
          "parrafo breve, usando UNICAMENTE la evidencia documental provista.\n"
          "Cenite al riesgo consultado: si un fragmento habla de OTRA cobertura (incendio, granizo, "
          "cristales, rescision), no la mezcles en la respuesta.\n"
          "REGLA INVIOLABLE: toda exclusion, condicion o requisito que la evidencia imponga sobre el "
          "riesgo consultado DEBE figurar en tu respuesta, aunque te parezca secundario. Nunca afirmes "
          "que no existen exclusiones: si la evidencia trae una, mencionala.\n"
          "No inventes numeros de articulo: cita uno solo si aparece escrito en la evidencia.")

resultados_e1 = {}
for clave, etiqueta in [("fijo", "💥 Índice roto"), ("con_solape", "✅ Índice sano")]:
    chunks_cfg, cfg, _ = indices_e1[clave]
    idx = IndiceRAG(chunks_cfg)
    recuperados = idx.denso(PREGUNTA_E1, top_k=3)   # 3 es el minimo para que la exclusion EXC-114
                                                   # entre en el contexto del indice sano
    contexto = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in recuperados)

    display(Markdown(f"---\n\n## {etiqueta}\n\nSegmentado con `{cfg['modo']} "
                     f"{cfg['chunk_size']}/{cfg['chunk_overlap']}`. Mismo modelo, mismo prompt, "
                     f"mismo Top-K que la otra corrida."))
    salida = ejecutar_consulta(
        prompt=f"EVIDENCIA DOCUMENTAL:\n{contexto}\n\nCONSULTA DEL ASEGURADO:\n{PREGUNTA_E1}\n\nRESPUESTA:",
        system_prompt=SYS_E1, temperature=0.1, contexto_referencia=contexto)

    auditoria = auditar_respuesta(salida["texto"], contexto, FICHA_E1)
    resultados_e1[clave] = {"auditoria": auditoria, "contexto": contexto,
                            "exc_en_contexto": "EXC-114" in contexto}

# ---------------------------- Veredicto consolidado ----------------------------
# Contrato de tablas del notebook: todas las celdas de una columna, del mismo tipo y largo.
# El veredicto largo NO entra en la tabla: baja a una linea propia debajo.
r_roto, r_sano = resultados_e1["fijo"], resultados_e1["con_solape"]

def _si_no(cond):
    return "🟢 SÍ" if cond else "🔴 NO"

tabla_e1 = f"""---

### ⚖️ El mismo cliente, la misma pregunta, dos índices

Lo único distinto entre las dos columnas es dónde cayeron los cortes durante la ingesta.

| | 💥 Índice roto | ✅ Índice sano |
| :--- | :---: | :---: |
| ¿La exclusión `EXC-114` llegó al contexto? | {_si_no(r_roto['exc_en_contexto'])} | {_si_no(r_sano['exc_en_contexto'])} |
| ¿La respuesta avisa del rastreo satelital? | {_si_no(not r_roto['auditoria']['terminos_faltantes'])} | {_si_no(not r_sano['auditoria']['terminos_faltantes'])} |
| Auditoría general (Experimento 3) | {semaforo(r_roto['auditoria'])} | {semaforo(r_sano['auditoria'])} |
"""

def _veredicto_e1(r):
    return ("promesa de pago sin la condición que la limita"
            if r["auditoria"]["terminos_faltantes"] else "dictamen completo y exigible")

tabla_e1 += f"""
**Veredicto —** índice roto: {_veredicto_e1(r_roto)}. Índice sano: {_veredicto_e1(r_sano)}.

> 🎯 **El punto del experimento:** el modelo generador, el prompt, la pregunta y el parámetro *Top-K* fueron **idénticos** en ambas corridas. Lo único que cambió fue **cómo se cortó el documento durante la ingesta** — una decisión tomada *offline*, semanas antes, por alguien que quizás nunca vio esta consulta. Un defecto de chunking no se manifiesta como un error del sistema: se manifiesta como una **promesa de pago que la compañía no está obligada a cumplir**.
"""
display(Markdown(tabla_e1))


In [ ]:
#@title 🧪 1.D — Exploración libre: mirá los fragmentos que produce tu configuración { display-mode: "form" }
import ipywidgets as widgets

# A diferencia de 1.A (que compara MÉTRICAS contra las tres anclas), acá el foco es el TEXTO:
# ver con tus propios ojos qué le llega al buscador cuando movés las perillas.

modo_w = widgets.Dropdown(options=[("Corte rígido por caracteres (fijo)", "fijo"),
                                   ("Recursivo por párrafos y oraciones", "recursivo")],
                          value="recursivo", description="Estrategia:",
                          style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
size_w = widgets.IntSlider(value=500, min=120, max=1200, step=20, description="Chunk Size:",
                           style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
ovl_w = widgets.IntSlider(value=100, min=0, max=300, step=10, description="Chunk Overlap:",
                          style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
par_w = widgets.Dropdown(options=[(p["nombre"], i) for i, p in enumerate(PARES_CRITICOS)],
                         value=0, description="Par crítico:",
                         style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
salida_w = widgets.Output()

def _marcar(chunk_texto, tramos):
    """Marca los tramos criticos DENTRO del bloque de codigo. No se usan negritas de markdown
       porque adentro de un bloque de codigo no se interpretan: van marcadores planos."""
    limpio = " ".join(chunk_texto.split())
    for fragmento, etiqueta in tramos:
        objetivo = " ".join(fragmento.split())
        if objetivo in limpio:
            limpio = limpio.replace(objetivo, f">>> {etiqueta} >>> {objetivo}")
    return limpio

def _bloque(titulo, chunk, tramos):
    return (f"**{titulo} — `{chunk['id']}`** · {chunk['longitud']} caracteres\n\n"
            f"```text\n{_marcar(chunk['texto'], tramos)}\n```\n\n")

def _render(_=None):
    with salida_w:
        salida_w.clear_output(wait=True)
        chs = segmentar_texto(DOCUMENTO_CORPORATIVO, chunk_size=size_w.value,
                              chunk_overlap=ovl_w.value, modo=modo_w.value)
        rad = radiografia_indice(chs)
        par = PARES_CRITICOS[par_w.value]

        # MISMO criterio que integridad_normativa(): el par esta intacto si EXISTE UN fragmento
        # que cubra el tramo completo, desde el inicio de la regla hasta el final de la excepcion.
        # (Buscar "el fragmento que contiene la regla" da falsos negativos cuando hay solape: con
        #  solape el tramo entero suele caer en el fragmento SIGUIENTE, que arranca mas atras.)
        ini_regla = DOCUMENTO_CORPORATIVO.find(par["regla"])
        fin_exc = DOCUMENTO_CORPORATIVO.find(par["excepcion"]) + len(par["excepcion"])
        completo = next((c for c in chs if c["ini"] <= ini_regla and c["fin"] >= fin_exc), None)

        t = (f"**{rad['n_chunks']} fragmentos** · solape `{100*ovl_w.value/max(1,size_w.value):.0f}%` "
             f"del chunk · integridad normativa **{rad['pares_intactos']}/{rad['pares_total']}** · "
             f"✂️ cortes limpios **{100 - rad['pct_mutilados']:.0f}%**\n\n"
             f"---\n\n### 🔍 `{par['nombre']}`\n\n")

        # Contrato de presentacion: el texto del fragmento va SIEMPRE en bloque de codigo
        # (es lo que recibe la maquina) y el comentario va afuera, como prosa.
        if completo is not None:
            t += _bloque("Un solo fragmento contiene las dos cosas", completo,
                         [(par["regla"], "REGLA"), (par["excepcion"], "EXCEPCION")])
            t += ("🟢 **La regla y su excepción viajan juntas.** El buscador no puede entregar la "
                  "promesa sin la condición que la limita.\n")
        else:
            c_regla = next((c for c in chs if c["ini"] <= ini_regla < c["fin"]), None)
            c_exc = next((c for c in chs if c["ini"] <= fin_exc - 1 < c["fin"]), None)
            if c_regla is not None:
                t += _bloque("Fragmento con la regla", c_regla, [(par["regla"], "REGLA")])
            if c_exc is not None and c_exc is not c_regla:
                t += _bloque("Fragmento con la excepción", c_exc, [(par["excepcion"], "EXCEPCION")])
            t += ("🔴 **Ningún fragmento contiene las dos.** El corte pasó entre la regla y su "
                  "excepción: para el buscador, la condición que limita el pago no existe.\n")

        display(Markdown(t))

for w in (modo_w, size_w, ovl_w, par_w):
    w.observe(_render, names="value")
display(widgets.VBox([modo_w, size_w, ovl_w, par_w, salida_w]))
_render()


### 📝 Registro de Análisis — Experimento 1 (Entregable TP2)

1. **El valor de la frontera.** ¿Con qué valor de solape el índice alcanzó `4/4` en la celda `1.B`? Expresalo como porcentaje del *chunk size* y explicá por qué ese número no es arbitrario: ¿qué propiedad del documento lo determina?
2. **La falla silenciosa.** En la celda `1.C`, ¿llegó la exclusión `EXC-114` al contexto en cada caso, y qué le respondió el sistema al asegurado? Describí el perjuicio concreto para la compañía si esa respuesta se emite en un canal oficial de autogestión.
3. **El trade-off que no se ve en la tabla.** El solape del 20% resolvió la integridad normativa, pero repite texto en varios fragmentos. Enumerá los tres costos que esa repetición introduce (indexación, almacenamiento y tokens en el prompt). ¿En qué escenario preferirías un *chunk* grande sin solape antes que uno chico con mucho solape?
4. **Traslado a tu dominio.** Pensá un documento real de tu organización —o de una que conozcas— que tenga la misma estructura "regla + excepción". ¿Qué par crítico definirías como métrica para auditar su ingesta?

> ✍️ **Tu análisis aquí:** *(Escribí acá tus observaciones y conclusiones para la entrega del TP2)*

## 🔬 Experimento 2 — Recuperación: tres motores y tres puntos ciegos distintos

### 🎯 El problema
Los buscadores corporativos clásicos fallan porque buscan **coincidencias literales**. Si alguien escribe *"¿cómo recupero la plata si doy de baja el seguro?"* y el manual dice *"compensación económica por rescisión contractual"*, un motor léxico devuelve cero resultados.

Pero la conclusión apresurada —"entonces usemos embeddings para todo"— es justamente el error que este experimento busca desarmar. **La búsqueda densa tiene su propio punto ciego, y es grave**: un modelo de embeddings puede deducir que hablás genéricamente de "problemas de red" y nunca devolverte la página del código exacto.

**Modelo mental — el mapa de significados y el índice alfabético:** el motor denso es un mapa donde las ideas afines quedan en el mismo vecindario; encuentra *"choque en autopista"* buscando *"colisión vehicular"*. El motor disperso (BM25) es el índice alfabético del final del libro: no entiende nada, pero si buscás `EXC-114` te lleva a la página exacta. **Son herramientas complementarias, no competidoras.**

### 🧪 Qué vamos a medir
Cuatro consultas, cada una con su **respuesta correcta anotada** (*ground truth*), elegidas para aislar un fenómeno distinto:

| | Consulta | Diseñada para que… |
| :--- | :--- | :--- |
| **Q1** | jerga coloquial, cero palabras en común con el manual | el denso acierte y **BM25 quede mudo** |
| **Q2** | un código de cláusula exacto, entre 9 códigos casi idénticos | BM25 acierte y **el denso se pierda** |
| **Q3** | granizo + franquicia, con 3 artículos compitiendo | ambos deban desambiguar |
| **Q4** | una pregunta cuya respuesta no existe en el corpus | **los tres fallen** |

### 📋 Consigna de exploración
1. **Ejecutá la celda `2.A`.** Mirá primero el marcador calibrado en **Top-K = 2**: ninguno de los dos motores puros gana las cuatro. Después bajá al detalle de **Q1** y **Q2**: son los dos extremos y muestran fallas de naturaleza opuesta. El formulario agrega un **segundo marcador** con el Top-K que elijas —antes de mirarlo, preguntate: ¿el motor mejoró, o simplemente le diste más intentos (y más contexto para pagar en tokens)?
2. Prestá atención a **qué devuelve BM25 en Q1**: no es un mal resultado, es *ningún* resultado. La pantalla en blanco del buscador corporativo.
3. En **Q2**, fijate qué recuperó el motor denso en primer lugar y con qué score. Un score "razonable" sobre el fragmento equivocado es más peligroso que un score bajo.
4. **Ejecutá la celda `2.B`** y probá la consulta `CLUST-90` a secas, y después alguna pregunta tuya en lenguaje coloquial. Es la forma más rápida de sentir la diferencia entre los motores.

In [ ]:
#@title 🚀 2.A — Duelo de motores: denso vs. BM25 vs. híbrido sobre cuatro consultas calibradas { display-mode: "form", run: "auto" }
# Mismo índice, mismos chunks, mismo Top-K. Lo único que cambia es CÓMO se busca.
# Cada consulta trae su respuesta correcta anotada (ground truth), así el acierto lo decide
# el código y no nuestra impresión al leer los fragmentos.
#
# El marcador con Top-K=2 (TOP_K_E2) es el ANCLA del experimento: se calcula siempre. El
# formulario de abajo agrega un SEGUNDO marcador, con el Top-K que elijas, para comparar.

#@markdown El marcador calibrado con **Top-K = 2** se calcula siempre. Este formulario agrega un **segundo marcador tuyo**:
top_k_tuyo = 6  #@param {type:"slider", min:1, max:6, step:1}

CONSULTAS_E2 = [
    {"id": "Q1", "tipo": "🗣️ Jerga coloquial",
     "texto": "Si doy de baja todo antes de que se termine, ¿me devuelven algo de lo que puse?",
     "gt": "ART-03", "porque": "El manual lo llama «rescisión contractual» y «restitución de primas». "
                               "La consulta no comparte NI UNA palabra clave con el artículo."},
    {"id": "Q2", "tipo": "🔢 Código exacto",
     "texto": "¿Qué establece exactamente la cláusula EXC-114?",
     "gt": "ART-05", "porque": "El corpus tiene 9 códigos con la misma forma (CLUST-90, RES-22, "
                               "ANX-7B, INC-57…). Para un embedding todos «significan» casi lo mismo."},
    {"id": "Q3", "tipo": "🎭 Distractor temático",
     "texto": "¿Qué franquicia se aplica si el granizo me rompe el parabrisas del auto?",
     "gt": "ART-10", "porque": "Hay granizo en 3 artículos y franquicias de 150, 300 y 90 USD. "
                               "Sólo una combinación es la correcta."},
    {"id": "Q4", "tipo": "🚫 Fuera de dominio",
     "texto": "¿Cuáles son los requisitos para un crédito hipotecario de vivienda?",
     "gt": None, "porque": "La respuesta NO existe en el corpus. Lo correcto es no devolver nada."},
]

MOTORES = [("denso", "🧠 Denso (embeddings)"), ("lexico", "🔤 BM25 (léxico)"), ("hibrido", "⚗️ Híbrido (RRF)")]
TOP_K_E2 = 2

def _correr_marcador(top_k):
    """Corre las 4 consultas con los 3 motores a un Top-K dado. Reutilizada por el ancla y por
       el bloque del estudiante, así ambas corridas son exactamente comparables."""
    resultados, puntajes, contexto_total = {}, {m: 0 for m, _ in MOTORES}, 0
    for q in CONSULTAS_E2:
        resultados[q["id"]] = {}
        for motor, _ in MOTORES:
            rs = indice.buscar(q["texto"], motor=motor, top_k=top_k)
            arts = [r["chunk"]["articulo"] for r in rs]
            acierto = (len(rs) == 0) if q["gt"] is None else (q["gt"] in arts)
            puntajes[motor] += int(acierto)
            resultados[q["id"]][motor] = {"rs": rs, "arts": arts, "acierto": acierto}
            if motor == "hibrido":
                contexto_total += sum(len(r["chunk"]["texto"]) for r in rs)
    return resultados, puntajes, contexto_total / len(CONSULTAS_E2)

def _celda_motor(r, gt):
    """Contenido de una celda del marcador.
       Si el motor ACIERTA, muestra en que puesto del ranking aparecio el articulo correcto:
       recuperarlo 1.o no es lo mismo que recuperarlo 4.o, y esa diferencia es exactamente la
       dimension 'Context Precision' del marco RAGAS. Mostrar siempre el primer articulo daba
       celdas contradictorias del tipo "OK ART-02" cuando el correcto era ART-05 y estaba 2.o.
       Si FALLA, muestra que trajo en primer lugar."""
    if not r["rs"]:
        return "✅ ∅ nada" if r["acierto"] else "❌ ∅ vacío"
    if r["acierto"] and gt is not None:
        return f"✅ {r['arts'].index(gt) + 1}.º"
    primero = r["arts"][0] or "—"        # los fragmentos de cabecera no pertenecen a ningun articulo
    return f"❌ `{primero}`"

resultados_e2, puntajes, ctx_prom_ancla = _correr_marcador(TOP_K_E2)

# Las consultas van FUERA de la tabla, como lista: metidas en una celda obligaban a la tabla a
# crecer a lo ancho y el texto quedaba cortado. En la tabla se las referencia por su codigo.
tabla = f"""### 🎯 Ningún motor puro gana las cuatro consultas

Mismo índice, mismos fragmentos, mismo Top-K = {TOP_K_E2}. Lo único que cambia es **cómo** se busca.

"""
for q in CONSULTAS_E2:
    tabla += f"* **{q['id']}** · {q['tipo']} — *«{q['texto']}»* → correcto: `{q['gt'] or 'ninguno'}`\n"

tabla += f"""
Cuando el motor acierta, la tabla dice **en qué puesto** del ranking apareció el artículo correcto; cuando falla, qué trajo en primer lugar.

| Consulta | 🧠 Denso | 🔤 BM25 | ⚗️ Híbrido |
| :--- | :---: | :---: | :---: |
"""
for q in CONSULTAS_E2:
    fila = f"| **{q['id']}** |"
    for motor, _ in MOTORES:
        fila += " " + _celda_motor(resultados_e2[q["id"]][motor], q["gt"]) + " |"
    tabla += fila + "\n"

# Sin linea en blanco antes del TOTAL: una linea vacia cierra la tabla en markdown y la fila
# quedaria suelta, como parrafo, debajo.
tabla += (f"| **TOTAL** | **{puntajes['denso']}/4** | **{puntajes['lexico']}/4** "
          f"| **{puntajes['hibrido']}/4** |\n")

tabla += """
---
#### 🔬 Los dos extremos, en detalle
"""
for qid in ["Q1", "Q2"]:
    q = next(x for x in CONSULTAS_E2 if x["id"] == qid)
    tabla += f"\n**{q['id']} — {q['tipo']}:** *«{q['texto']}»*  \n*{q['porque']}*\n\n"
    # La tabla se queda solo con lo corto y alineable. El texto del fragmento es evidencia:
    # va debajo, en bloque de codigo, como en todo el resto del notebook.
    tabla += "| Motor | Resultado | Score (1.º) | Artículo (1.º) |\n| :--- | :--- | :--- | :---: |\n"
    for motor, nombre in MOTORES:
        r = resultados_e2[qid][motor]
        if not r["rs"]:
            tabla += f"| {nombre} | ❌ cero resultados | — | — |\n"
        else:
            top = r["rs"][0]
            # La marca sale de _celda_motor (igual que la tabla resumen): mide el Top-K y dice en
            # que puesto quedo el articulo correcto. Score y articulo son siempre los del 1.o.
            tabla += (f"| {nombre} | {_celda_motor(r, q['gt'])} | `{top['score']:.4f}` "
                      f"| `{top['chunk']['articulo'] or '—'}` |\n")
    if qid == "Q2":
        # Evidencia directa de "para un embedding, los 9 codigos casi 'significan' lo mismo":
        # el ranking denso COMPLETO, no solo el 1.o. Mismo pool (8) que usa hibrido() por defecto.
        rank_denso = indice.denso(q["texto"], top_k=8)
        tabla += "\n**🧠 El ranking denso completo** — todos los candidatos casi empatan:\n\n"
        tabla += "| Puesto | Chunk | Artículo | Score |\n| :---: | :--- | :---: | ---: |\n"
        for pos, r in enumerate(rank_denso, 1):
            art = r["chunk"]["articulo"] or "—"
            marca = " 🎯" if art == q["gt"] else ""
            tabla += f"| {pos} | `{r['chunk']['id']}` | `{art}`{marca} | {r['score']:.4f} |\n"

        # Como arma RRF el score hibrido: NO promedia scores (no son comparables entre motores),
        # promedia PUESTOS. El score de cada fila es el mismo que ya calculo el motor arriba.
        hib_pool = indice.hibrido(q["texto"], top_k=8)
        d_pool = {r["chunk"]["id"]: p for p, r in enumerate(rank_denso, 1)}
        l_pool = {r["chunk"]["id"]: p for p, r in enumerate(indice.lexico(q["texto"], top_k=8), 1)}
        tabla += ("\n**⚗️ Cómo arma RRF el score híbrido** — no promedia scores: suma, por cada "
                  "ranking donde el chunk aparece, un puntaje que premia el primer puesto y decae "
                  "con la posición:\n\n")
        tabla += "| Chunk | Artículo | Puesto denso | Puesto BM25 | Score RRF |\n| :--- | :---: | :---: | :---: | ---: |\n"
        for r in hib_pool[:3]:
            cid, art = r["chunk"]["id"], r["chunk"]["articulo"] or "—"
            marca = " 🎯" if art == q["gt"] else ""
            tabla += (f"| `{cid}` | `{art}`{marca} | {d_pool.get(cid, '—')} "
                      f"| {l_pool.get(cid, '—')} | {r['score']:.4f} |\n")

    tabla += "\nLo que cada motor puso en primer lugar:\n\n"
    for motor, nombre in MOTORES:
        r = resultados_e2[qid][motor]
        if not r["rs"]:
            tabla += f"{nombre} — *el buscador devuelve una página en blanco.*\n\n"
        else:
            top = r["rs"][0]
            extracto = " ".join(top["chunk"]["texto"][:220].split())
            tabla += (f"{nombre} — `{top['chunk']['id']}`\n\n"
                      f"```text\n{extracto}…\n```\n\n")

tabla += """
---
#### 📈 Puntaje según Top-K — ¿mejora el motor, o le diste más intentos?

Mismas 4 consultas, mismos 3 motores: lo único que cambia es cuántos candidatos se le piden a
cada uno. El ancla del experimento usa Top-K = 2 (marcado con ⚓).

"""
tabla += "| Top-K | 🧠 Denso | 🔤 BM25 | ⚗️ Híbrido |\n| :---: | :---: | :---: | :---: |\n"
barrido = {}
for k in range(1, 7):
    punt_k = puntajes if k == TOP_K_E2 else _correr_marcador(k)[1]   # el ancla no se recalcula dos veces
    barrido[k] = punt_k
    ancla_marca = " ⚓" if k == TOP_K_E2 else ""
    tabla += f"| {k}{ancla_marca} | {punt_k['denso']}/4 | {punt_k['lexico']}/4 | {punt_k['hibrido']}/4 |\n"

# El barrido decide con datos si el hibrido "gana o empata siempre" — no se afirma a ojo, porque
# los scores del denso pueden moverse un poco entre corridas (CPU vs GPU) y correr filas de lugar.
hibrido_no_pierde = all(p["hibrido"] >= max(p["denso"], p["lexico"]) for p in barrido.values())
frase_barrido = ("gana o empata contra los dos motores puros en las seis filas del barrido"
                  if hibrido_no_pierde else
                  "no gana siempre — mirá en qué fila del barrido queda por debajo de algún motor puro")

# El cierre de Q2 tambien sale del resultado YA calculado, no de un numero escrito a mano: si el
# ranking denso corre de lugar en otra corrida, el texto no puede quedar desactualizado.
gt_q2 = next(x for x in CONSULTAS_E2 if x["id"] == "Q2")["gt"]
r_hib_q2 = resultados_e2["Q2"]["hibrido"]
puesto_hib_q2 = r_hib_q2["arts"].index(gt_q2) + 1 if gt_q2 in r_hib_q2["arts"] else None
if puesto_hib_q2 == 1:
    nota_q2 = "En Q2 el consenso entre motores ya alcanza para poner el correcto 1.º."
elif puesto_hib_q2:
    nota_q2 = (f"En Q2 el híbrido lo deja {puesto_hib_q2}.º, no 1.º: fusiona el CONSENSO entre "
               f"motores, no rescata automáticamente al que un solo motor acertó con fuerza.")
else:
    nota_q2 = "En Q2 el híbrido ni siquiera lo trae dentro del Top-K del ancla."

tabla += f"""
> ⚠️ **Los scores no son comparables entre filas:** el denso informa similitud coseno (0 a 1),
> BM25 informa un puntaje sin techo y el híbrido un puntaje de fusión por *ranking*. Lo comparable
> es **el acierto**, no el número.
>
> 🎯 **Por qué la búsqueda híbrida es el estándar de producción:** cada motor tiene un punto ciego
> y son **puntos ciegos complementarios**. El denso entiende la intención pero no distingue un
> código de otro; BM25 clava el código exacto pero se queda mudo si el usuario no usa las palabras
> del manual. RRF fusiona por posición en el ranking — no promedia scores, promedia PUESTOS — y en
> este corpus {frase_barrido}.
>
> ⚠️ **Pero RRF no "hereda lo mejor de ambos" de forma automática — mirá Q2:** {nota_q2} En la tabla
> de fusión de arriba, el fragmento equivocado gana por ser 1.º en un motor y 2.º en el otro,
> mientras el correcto es 1.º en uno pero 4.º en el otro: el consenso débil de los dos le gana a la
> certeza fuerte de uno solo.
>
> 🚧 **Y ninguno resuelve Q4:** los tres devuelven *algo* ante una pregunta cuya respuesta no existe.
> Ese problema no se arregla buscando mejor — se arregla **sabiendo cuándo no responder** (Experimento 4).
"""
display(Markdown(tabla))

# --- Segundo marcador: TU Top-K, con el contrapeso del contexto inyectado ----------------------
resultados_tuyos, puntajes_tuyos, ctx_prom_tuyo = _correr_marcador(top_k_tuyo)

tabla_tuya = f"""---

### 👤 El mismo duelo, con tu Top-K = {top_k_tuyo}

Mismos motores y mismas consultas: lo único que cambia es cuántos fragmentos se le pide a cada uno.

| Consulta | 🧠 Denso | 🔤 BM25 | ⚗️ Híbrido |
| :--- | :---: | :---: | :---: |
"""
for q in CONSULTAS_E2:
    fila = f"| **{q['id']}** |"
    for motor, _ in MOTORES:
        fila += " " + _celda_motor(resultados_tuyos[q["id"]][motor], q["gt"]) + " |"
    tabla_tuya += fila + "\n"

tabla_tuya += (f"| **TOTAL** | **{puntajes_tuyos['denso']}/4** | **{puntajes_tuyos['lexico']}/4** "
               f"| **{puntajes_tuyos['hibrido']}/4** |\n")

tabla_tuya += f"""

| | Top-K = {TOP_K_E2} (ancla) | Top-K = {top_k_tuyo} (tuyo) |
| :--- | ---: | ---: |
| Contexto inyectado, híbrido (promedio) | {ctx_prom_ancla:.0f} car. (~{ctx_prom_ancla/4:.0f} tok) | {ctx_prom_tuyo:.0f} car. (~{ctx_prom_tuyo/4:.0f} tok) |
"""

denom = max(1.0, ctx_prom_ancla)
if puntajes_tuyos["hibrido"] > puntajes["hibrido"] and ctx_prom_tuyo <= ctx_prom_ancla * 1.5:
    veredicto2 = (f"👤 Con `Top-K={top_k_tuyo}` el híbrido mejora a **{puntajes_tuyos['hibrido']}/4** "
                  f"sin inflar demasiado el contexto: acá subir el Top-K ayudó de verdad.")
elif puntajes_tuyos["hibrido"] > puntajes["hibrido"]:
    veredicto2 = (f"👤 Con `Top-K={top_k_tuyo}` el híbrido sube a **{puntajes_tuyos['hibrido']}/4**, pero "
                  f"el contexto inyectado creció **{ctx_prom_tuyo/denom:.1f}×**: ¿mejoró el motor, o "
                  f"le diste más intentos (y más ruido para pagar en tokens)?")
elif puntajes_tuyos["hibrido"] == puntajes["hibrido"] and ctx_prom_tuyo > ctx_prom_ancla * 1.2:
    veredicto2 = (f"👤 Con `Top-K={top_k_tuyo}` el híbrido se mantiene en **{puntajes_tuyos['hibrido']}/4**, "
                  f"pero el contexto creció **{ctx_prom_tuyo/denom:.1f}×**: pagaste más tokens sin ganar acierto.")
elif puntajes_tuyos["hibrido"] == puntajes["hibrido"]:
    veredicto2 = f"👤 Con `Top-K={top_k_tuyo}` el híbrido se mantiene en **{puntajes_tuyos['hibrido']}/4**, sin cambios de fondo."
else:
    veredicto2 = (f"👤 Con `Top-K={top_k_tuyo}` el híbrido empeora a **{puntajes_tuyos['hibrido']}/4** "
                  f"(vs. {puntajes['hibrido']}/4 del ancla): más candidatos también pueden diluir el ranking.")

tabla_tuya += f"\n> {veredicto2}\n"
display(Markdown(tabla_tuya))


In [ ]:
#@title 🧪 2.B — Exploración libre: escribí tu consulta y compará los tres motores { display-mode: "form" }
import ipywidgets as widgets

sugerencias = [q["texto"] for q in CONSULTAS_E2] + [
    "¿Qué pasa si manejaba con unas copas de más?",
    "Se me rompió el parabrisas, ¿cuánto pago yo?",
    "¿Me cubren si viajo a Brasil de vacaciones?",
    "CLUST-90",
]
sel_w = widgets.Dropdown(options=sugerencias, value=sugerencias[0], description="Sugerencias:",
                         style={'description_width': 'initial'}, layout=widgets.Layout(width='85%'))
libre_w = widgets.Text(value="", placeholder="(o escribí tu propia consulta acá)",
                       description="Tu consulta:", style={'description_width': 'initial'},
                       layout=widgets.Layout(width='85%'))
k_w = widgets.IntSlider(value=2, min=1, max=5, step=1, description="Top-K:",
                        style={'description_width': 'initial'}, layout=widgets.Layout(width='45%'))
btn = widgets.Button(description="🔍 Comparar motores", button_style="primary")
out_w = widgets.Output()

def _buscar(_=None):
    with out_w:
        out_w.clear_output(wait=True)
        q = libre_w.value.strip() or sel_w.value
        t = f"### 🔍 *«{q}»* · Top-K = {k_w.value}\n\n"
        for motor, nombre in MOTORES:
            rs = indice.buscar(q, motor=motor, top_k=k_w.value)
            t += f"**{nombre}**\n\n"
            if not rs:
                t += "> ⚠️ *Cero resultados: el buscador devuelve una página en blanco.*\n\n"
                continue
            # El ranking va en tabla (celdas cortas y alineables) y el texto de cada fragmento
            # debajo, en bloque de codigo: metido en una celda se salia del ancho y se cortaba.
            t += "| # | Chunk | Artículo | Score |\n| ---: | :--- | :---: | ---: |\n"
            for i, r in enumerate(rs, 1):
                t += (f"| {i} | `{r['chunk']['id']}` | `{r['chunk']['articulo'] or '—'}` "
                      f"| `{r['score']:.4f}` |\n")
            t += "\n"
            for i, r in enumerate(rs, 1):
                ex = " ".join(r["chunk"]["texto"][:240].split())
                t += f"`{i}` · `{r['chunk']['id']}`\n\n```text\n{ex}…\n```\n\n"
        t += ("\n> 💡 Probá `CLUST-90` a secas: BM25 lo clava por coincidencia exacta y el motor "
              "denso se pierde entre los nueve códigos casi idénticos del manual.\n")
        display(Markdown(t))

btn.on_click(_buscar)
display(widgets.VBox([sel_w, libre_w, k_w, btn, out_w]))
_buscar()

### 📝 Registro de Análisis — Experimento 2 (Entregable TP2)

1. **Anatomía de dos fracasos opuestos.** Transcribí el marcador final (denso / BM25 / híbrido). Explicá con tus palabras por qué BM25 devuelve **cero** resultados en Q1 y por qué el motor denso **no distingue** `EXC-114` del resto de los códigos en Q2. Son dos limitaciones de naturaleza distinta: nombralas.
2. **El argumento para la búsqueda híbrida.** Si tuvieras que justificar ante un comité de arquitectura el costo extra de mantener **dos índices en paralelo** más una fusión RRF, ¿qué evidencia de esta tabla usarías? ¿Qué tipo de consulta de tu negocio justificaría la inversión?
3. **Impacto operativo del Top-K.** Volvé a la celda `2.B` y subí el Top-K de 2 a 5 en una consulta. ¿Qué le pasa a la precisión de lo recuperado? Relacionalo con el costo en tokens, la latencia y el fenómeno *"Lost in the Middle"* (pérdida de atención en contextos largos).
4. **Lo que ningún motor resolvió.** Los tres fallaron en Q4. ¿Por qué este problema **no se arregla mejorando el buscador**? Anticipá qué mecanismo haría falta (lo construimos en el Experimento 4).

> ✍️ **Tu análisis aquí:** *(Escribí acá tus observaciones y conclusiones para la entrega del TP2)*

## 🔬 Experimento 3 — Generación fundamentada: cuándo responder y cuándo callarse

### 🎯 El problema
Un LLM puro rinde examen **a libro cerrado**: responde con lo que quedó en sus parámetros. Cuando desconoce una normativa privada no dice "no sé" — **completa el hueco con una respuesta verosímil**. Como se vio en la teoría, el modelo no distingue con claridad entre lo que le dieron en la entrada y lo que su propio decodificador va generando (*self-delusion*).

Un sistema RAG opera **a libro abierto**: tiene la evidencia delante y su tarea es sintetizar **citando la fuente**. Y si la evidencia no alcanza, la arquitectura le impone **abstenerse**.

### 🧪 Por qué evaluamos con dos preguntas y no con una
Este es el punto metodológico del experimento. **Con una sola pregunta fácil, el RAG ingenuo y el RAG fundamentado producen resultados casi idénticos** y el ejercicio no demuestra nada: si el contexto recuperado ya contiene la respuesta, el modelo acierta con reglas o sin ellas.

La diferencia aparece cuando la respuesta **no está**. Por eso evaluamos una matriz de **3 arquitecturas × 2 preguntas**:

* **📗 P1 — respondible:** el Artículo 2 la contesta completa (72 horas + tres documentos).
* **📕 P2 — sin respuesta:** el manual cubre el robo total, pero **no fija ninguna franquicia** para ese riesgo. Sin embargo, el corpus **sí tiene** franquicias de 150, 300 y 90 USD… de otros riesgos. La trampa está tendida: el modelo tiene cifras a mano, todas incorrectas para esta pregunta.

Cada respuesta se audita automáticamente detectando **cifras que no están en el contexto** (alucinación), **cifras trasladadas de otro artículo** (contaminación), **citas a fragmentos inexistentes** y **abstención correcta**.

### 📋 Consigna de exploración
1. **Ejecutá la celda `3.A`.** Son seis generaciones; tomará un par de minutos.
2. Leé la matriz final de veredictos: **cada fila tiene una firma distinta**. Si dos filas te quedaron iguales, anotá en qué columna se diferencian igual (mirá la de trazabilidad).
3. Prestá especial atención a la celda **RAG ingenuo × P2**: es el caso que mejor explica por qué un sistema que "viene funcionando bien" puede fallar catastróficamente ante una pregunta nueva.
4. **Ejecutá la celda `3.B`** y probá preguntas propias. Buscá una donde el modo fundamentado se abstenga aunque vos sepas que la respuesta está en el manual: eso es un **falso negativo**, y también es un costo del diseño.

In [ ]:
#@title 🚀 3.A — Matriz de fundamentación: 3 arquitecturas × 2 preguntas, auditadas automáticamente { display-mode: "form" }
# Una sola pregunta no alcanza para distinguir las tres arquitecturas: con una consulta fácil,
# el RAG ingenuo y el fundamentado aciertan igual. Por eso evaluamos DOS preguntas de naturaleza
# opuesta: una que el manual SÍ responde y otra cuya respuesta NO existe en ningún artículo.

PREGUNTAS_E3 = [
    {"id": "P1 · respondible", "icono": "📗",
     "texto": "Choqué en la autopista. ¿En qué plazo tengo que denunciarlo y qué papeles llevo?",
     "ficha": {"cifras_esperadas": ["72"], "cifras_prohibidas": [],
               "debe_abstenerse": False, "terminos_obligatorios": []},
     "nota": "El Artículo 2 la responde completa: 72 horas hábiles + tres documentos."},
    {"id": "P2 · sin respuesta", "icono": "📕",
     "texto": "¿Cuál es la franquicia deducible que debo pagar por el robo total de mi auto?",
     "ficha": {"cifras_esperadas": [], "cifras_prohibidas": ["150", "300", "90"],
               "debe_abstenerse": True, "terminos_obligatorios": []},
     "nota": ("TRAMPA: el Artículo 5 cubre robo total pero **no fija ninguna franquicia**. "
              "Las cifras 150, 300 y 90 USD existen en el manual, pero pertenecen a "
              "*otros* riesgos (clima, incendio, cristales). Usarlas acá es contaminación de contexto.")},
]

SYS_PURO = "Eres un asesor oficial de atencion al cliente de la aseguradora Vertex Horizon Seguros."

SYS_INGENUO = ("Eres un asistente de seguros. Responde la pregunta del usuario utilizando la "
               "informacion provista si te resulta util.")

SYS_GROUNDED = """Actuas como Liquidador Legal Senior de Siniestros de Vertex Horizon Seguros.

REGLAS DE OPERACION, EN ORDEN DE PRIORIDAD:

1. VERIFICACION PREVIA (regla de maxima prioridad). Antes de redactar, comproba si el dato exacto
   que se consulta figura LITERALMENTE en la evidencia. Que la evidencia hable del mismo tema NO
   alcanza: si preguntan por la franquicia de un riesgo y la evidencia solo trae la franquicia de
   OTRO riesgo, el dato NO figura. Esta PROHIBIDO deducir, estimar o trasladar cifras de un
   articulo a otro.
   Si el dato no figura, responde EXACTAMENTE esta frase, sin citas y sin agregar nada mas:
   'No poseo informacion oficial en los manuales suministrados para responder esta consulta'

2. Si el dato SI figura, responde unicamente con hechos explicitos de la evidencia.

3. CITA OBLIGATORIA (aplica solo cuando respondes por la regla 2). Cada afirmacion cierra con el
   identificador real del fragmento que la respalda. Formato, con un ejemplo generico que NO
   pertenece a este manual y que solo ilustra la forma:
   "El tramite exige el formulario A [FUENTE: CHUNK_03]. El plazo vence a los cinco dias
   [FUENTE: CHUNK_04]."
   La frase de abstencion de la regla 1 es la UNICA respuesta valida que va sin cita.

4. COMPLETITUD: responde TODO lo que se consulta. Si preguntan por un plazo y por la documentacion
   a presentar, tu respuesta debe cubrir ambas cosas, cada una con su cita. Se conciso: como
   maximo cuatro oraciones, pero nunca a costa de dejar afuera parte de la pregunta."""

ARQUITECTURAS = [
    ("1️⃣ Modelo puro", "libro cerrado, sin documentos"),
    ("2️⃣ RAG ingenuo", "con contexto, sin reglas"),
    ("3️⃣ RAG fundamentado", "con citas y abstención"),
]

matriz_e3 = {}
for preg in PREGUNTAS_E3:
    display(Markdown(f"---\n# {preg['icono']} {preg['id']}\n> *«{preg['texto']}»*\n>\n> {preg['nota']}"))
    matriz_e3[preg["id"]] = {}
    for arq, _ in ARQUITECTURAS:
        if arq.startswith("1"):
            contexto, sys_p = "", SYS_PURO
            user_p = preg["texto"]
        else:
            rec = indice.denso(preg["texto"], top_k=3)
            if arq.startswith("2"):
                contexto = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in rec)
                sys_p = SYS_INGENUO
                user_p = f"Informacion:\n{contexto}\n\nPregunta: {preg['texto']}"
            else:
                contexto = "\n\n".join(
                    f"[FUENTE: {r['chunk']['id']} | relevancia {r['score']:.2f}]\n{r['chunk']['texto']}"
                    for r in rec)
                sys_p = SYS_GROUNDED
                user_p = (f"EVIDENCIA DOCUMENTAL OFICIAL:\n{contexto}\n\n"
                          f"CONSULTA DEL ASEGURADO:\n{preg['texto']}\n\n"
                          f"DICTAMEN FUNDAMENTADO (cada afirmacion con su [FUENTE: CHUNK_XX]):")

        display(Markdown(f"### {arq}"))
        salida = ejecutar_consulta(prompt=user_p, system_prompt=sys_p, temperature=0.1,
                                   contexto_referencia=contexto)
        aud = auditar_respuesta(salida["texto"], contexto, preg["ficha"])
        matriz_e3[preg["id"]][arq] = {"aud": aud, "tok_in": salida["tok_in"], "ttft": salida["ttft"]}

# ---------------------------- Veredicto consolidado ----------------------------
# Contrato de tablas del notebook: nada de HTML (saltos de linea ni subindices)
# dentro de las celdas. La descripcion de cada arquitectura y el detalle de cada fallo
# bajan a lineas propias debajo de la tabla.
tabla = """### 🎯 Cada arquitectura tiene una firma distinta

Mismo modelo, mismo índice, las mismas dos preguntas. Lo único que cambia entre las filas es de cuánto contexto dispone el modelo y con qué reglas opera.

| Arquitectura | 📗 P1 · pregunta respondible | 📕 P2 · pregunta sin respuesta | 🔗 ¿Auditable? |
| :--- | :--- | :--- | :--- |
"""
detalles = []
for arq, desc in ARQUITECTURAS:
    a1 = matriz_e3[PREGUNTAS_E3[0]["id"]][arq]["aud"]
    a2 = matriz_e3[PREGUNTAS_E3[1]["id"]][arq]["aud"]
    traz = "🟢 sí, con citas" if a1["trazable"] else "🔴 no, sin citas verificables"
    tabla += f"| **{arq}** | {semaforo(a1)} | {semaforo(a2)} | {traz} |\n"
    obs = [f"*{desc}*"]
    if a1["cifras_inventadas"]:
        obs.append(f"en P1 inventa {a1['cifras_inventadas']}")
    if a2["cifras_prohibidas"]:
        obs.append(f"en P2 usa una cifra ajena {a2['cifras_prohibidas']}")
    elif a2["se_abstuvo"]:
        obs.append("en P2 se abstuvo")
    detalles.append(f"* **{arq}** — {' · '.join(obs)}")
tabla += "\n" + "\n".join(detalles) + "\n"

tabla += """
> 🔍 **Cómo leer la matriz — cada fila tiene una firma distinta:**
>
> * **Modelo puro:** falla en ambas. Sin documentos no tiene con qué responder, así que **llena el hueco igual**: a veces con una cifra inventada con toda naturalidad (un teléfono de contacto, una tabla de franquicias del 0 al 15%), a veces con una respuesta genérica que nunca llega al dato. Es el incidente que abrió la clase.
> * **RAG ingenuo:** acierta P1 y **falla P2**. Es el caso peligroso: parece funcionar hasta que llega una pregunta cuya respuesta no está. Y aun cuando acierta, lo hace **sin citas**: nadie puede auditar de dónde salió el dato.
> * **RAG fundamentado:** acierta ambas. La regla de abstención convierte *«no sé»* en una respuesta válida del sistema, y la cita obligatoria hace cada afirmación **verificable**.
>
> ⚠️ **Mirá con atención qué hizo el RAG ingenuo en P2**, porque el modo de fallar cambia de corrida en corrida y el semáforo te dice cuál te tocó:
>
> * `🔴 ALUCINACION POR CONTAMINACION` — tomó una franquicia de *otro* riesgo (150, 300 o 90 USD) y la presentó como propia.
> * `🔴 NO SE ABSTUVO` — **detectó el vacío pero no se detuvo**. Suele escribir *"no hay una franquicia específica mencionada"* y, acto seguido, concluir *"por lo tanto no tenés que pagar franquicia"*. Eso ya no está en el manual: es una inferencia propia presentada como dictamen.
>
> Las dos son la misma falla de fondo —**llenar un vacío en vez de reportarlo**— y las dos terminan en una afirmación que la compañía no puede sostener. La segunda es la más traicionera, porque el razonamiento *suena* correcto: el modelo parece estar siendo prudente justo antes de afirmar lo que no sabe.
>
> 💡 **La diferencia entre la fila 2 y la fila 3 no es el modelo ni los documentos: es el prompt.**
> Mismo LLM, mismo índice, misma pregunta — sólo cambian las reglas de operación.
"""
display(Markdown(tabla))

In [ ]:
#@title 🧪 3.B — Exploración libre: tu pregunta contra la arquitectura que elijas { display-mode: "form" }
import ipywidgets as widgets

arq_w = widgets.Dropdown(options=["1️⃣ Modelo puro (libro cerrado)",
                                  "2️⃣ RAG ingenuo (contexto sin reglas)",
                                  "3️⃣ RAG fundamentado (citas + abstención)"],
                         value="3️⃣ RAG fundamentado (citas + abstención)", description="Arquitectura:",
                         style={'description_width': 'initial'}, layout=widgets.Layout(width='65%'))
preg_w = widgets.Textarea(value="¿Qué pasa si denuncio el choque cinco días después?",
                          description="Pregunta:", rows=2,
                          style={'description_width': 'initial'}, layout=widgets.Layout(width='92%'))
temp_w = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description="Temperature:",
                             style={'description_width': 'initial'}, layout=widgets.Layout(width='55%'))
btn3 = widgets.Button(description="⚡ Generar respuesta", button_style="primary")
out3 = widgets.Output()

def _generar(_=None):
    with out3:
        out3.clear_output(wait=True)
        q = preg_w.value.strip()
        if arq_w.value.startswith("1"):
            ctx, sys_p, user_p = "", SYS_PURO, q
        else:
            rec = indice.denso(q, top_k=3)
            if arq_w.value.startswith("2"):
                ctx = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in rec)
                sys_p, user_p = SYS_INGENUO, f"Informacion:\n{ctx}\n\nPregunta: {q}"
            else:
                ctx = "\n\n".join(f"[FUENTE: {r['chunk']['id']} | relevancia {r['score']:.2f}]\n"
                                  f"{r['chunk']['texto']}" for r in rec)
                sys_p = SYS_GROUNDED
                user_p = (f"EVIDENCIA DOCUMENTAL OFICIAL:\n{ctx}\n\nCONSULTA DEL ASEGURADO:\n{q}\n\n"
                          f"DICTAMEN FUNDAMENTADO:")
        salida = ejecutar_consulta(prompt=user_p, system_prompt=sys_p,
                                   temperature=temp_w.value, contexto_referencia=ctx)
        aud = auditar_respuesta(salida["texto"], ctx, {"cifras_esperadas": [], "cifras_prohibidas": [],
                                                       "debe_abstenerse": False})
        alerta = (f"🔴 **Cifras que NO están en el contexto: {aud['cifras_inventadas']}**"
                  if aud["cifras_inventadas"] else "🟢 Todas las cifras citadas figuran en el contexto.")
        traza = (f"🟢 Citas verificables: `{', '.join(aud['citas_validas'])}`" if aud["citas_validas"]
                 else "🟠 Sin citas verificables: la afirmación no es auditable.")
        display(Markdown(f"#### 🔎 Auditoría automática\n\n* {alerta}\n* {traza}"))

btn3.on_click(_generar)
display(widgets.VBox([arq_w, preg_w, temp_w, btn3, out3]))

### 📝 Registro de Análisis — Experimento 3 (Entregable TP2)

1. **Diagnóstico comparado.** Transcribí la matriz de veredictos. ¿Qué cifra inventó el modelo puro en P1? En P2, ¿qué franquicia tomó el RAG ingenuo y **de qué artículo la sacó**? Explicá por qué este error es más peligroso que una alucinación evidente.
2. **La abstención como decisión de diseño.** El modo fundamentado responde *"no poseo información oficial"* ante P2. Para un área comercial eso puede sonar a que "el bot no sirve". Construí el argumento que usarías frente a esa objeción, incorporando el riesgo legal y reputacional.
3. **Trazabilidad y auditoría.** El RAG ingenuo acertó P1 pero **sin citas**. Si seis meses después un asegurado inicia un reclamo por esa respuesta, ¿qué diferencia operativa concreta hay entre tener y no tener el identificador del fragmento que la respaldó?
4. **El prompt como capa de control.** Entre la fila 2 y la fila 3 no cambió el modelo, ni el índice, ni la pregunta: cambió el *system prompt*. ¿Qué implica esto sobre dónde reside realmente la confiabilidad de un sistema RAG? Relacionalo con lo visto en el Laboratorio 2.

> ✍️ **Tu análisis aquí:** *(Escribí acá tus observaciones y conclusiones para la entrega del TP2)*

## 🔬 Experimento 4 — Corrective RAG: el filtro que protege y lo que se le escapa

### 🎯 El problema
En un RAG ingenuo el recuperador entrega ciegamente los *Top-K* fragmentos más cercanos **aunque todos sean irrelevantes**. No existe el resultado "no encontré nada": siempre hay un vecino más cercano, por lejano que esté.

**Modelo mental — el cadete y el revisor técnico:** el cadete baja al archivo, agarra los dos primeros folletos que encuentra y se los tira al perito sobre el escritorio: *"escribí con esto"*. Si la consulta era sobre hipotecas y los folletos eran de autos, el perito igual va a intentar redactar algo. **CRAG agrega un revisor intermedio** que evalúa la calidad de lo recuperado *antes* de escribir: si nada supera el umbral de confianza, descarta todo y activa una ruta segura.

### 🧪 Qué vamos a medir
Dos preguntas sin respuesta en el corpus, pero de naturaleza **muy distinta**:

* **🚫 Pregunta A — otro dominio:** un crédito hipotecario. Nada que ver con el manual; similitud baja.
* **🎭 Pregunta B — dominio correcto, dato inexistente:** la franquicia por robo de auto. Habla de autos, de robo y de franquicias: **todo suena a este manual**, y la similitud es alta. Pero el dato no existe.

Esa segunda pregunta es la interesante, porque expone el límite real del patrón: **el umbral mide parecido temático, no presencia del dato**.

### 📋 Consigna de exploración
1. **Ejecutá la celda `4.A`.** Compará lo que hace CRAG con la pregunta A y con la pregunta B. Registrá la similitud máxima de cada una: ahí está toda la explicación.
2. Fijate en la telemetría de la vía bloqueada: los tokens que **no** se enviaron son ahorro real y medido, no una estimación.
3. **Ejecutá la celda `4.B`.** Es el barrido del umbral sobre un set etiquetado de seis consultas. Buscá los dos extremos: el umbral que deja pasar todo y el que bloquea hasta las consultas legítimas. Anotá el punto de equilibrio.
4. **Ejecutá la celda `4.C`** y movés el umbral a mano para ver la bifurcación en vivo.

In [ ]:
#@title 🚀 4.A — CRAG frente a dos preguntas sin respuesta: una que filtra y otra que se le escapa { display-mode: "form" }
# El evaluador de CRAG mide UNA cosa: cuánto se parece la pregunta a lo que hay indexado.
# Eso lo vuelve excelente contra preguntas de otro dominio... y ciego ante preguntas del
# dominio correcto cuyo dato puntual no existe. Medimos exactamente esa frontera.

UMBRAL_E4 = 0.50   # calibrado contra el set etiquetado de la celda 4.B (ver también 0.5)

PREGUNTAS_E4 = [
    {"id": "A", "clase": "🚫 Otro dominio",
     "texto": "¿Cuáles son los requisitos y la tasa de interés para un crédito hipotecario a 20 años?",
     "espera": "El corpus no tiene una sola línea sobre créditos. Debería bloquearse."},
    {"id": "B", "clase": "🎭 Dominio correcto, dato inexistente",
     "texto": "¿Cuál es la franquicia deducible que debo pagar por el robo total de mi auto?",
     "espera": "Habla de autos, robo y franquicias: todo suena a este manual. Pero el Artículo 5 "
               "**no fija franquicia alguna** para robo."},
]

FICHA_E4 = {"cifras_esperadas": [], "cifras_prohibidas": ["150", "300", "90"],
            "debe_abstenerse": True, "terminos_obligatorios": []}

MSG_CRAG = ("🛑 **ALERTA CRAG — evidencia insuficiente.** Ningún fragmento del corpus alcanzó el "
            "umbral de relevancia. Se aborta la llamada al LLM para no inducir una alucinación.")

resumen_e4 = []
for q in PREGUNTAS_E4:
    recuperados = indice.denso(q["texto"], top_k=3)
    sim_max = recuperados[0]["score"] if recuperados else 0.0
    aprobados = [r for r in recuperados if r["score"] >= UMBRAL_E4]
    contexto = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in recuperados)

    display(Markdown(
        f"---\n# Pregunta {q['id']} — {q['clase']}\n> *«{q['texto']}»*\n>\n> {q['espera']}\n\n"
        f"**Evaluador de recuperación:** similitud máxima `{sim_max:.4f}` vs. umbral `{UMBRAL_E4:.2f}` → "
        f"{'🟢 pasa el filtro' if aprobados else '🔴 todo descartado'}"))

    # --- Vía 1: Naive RAG, inyecta lo que haya ---
    display(Markdown("### ⚠️ Naive RAG (inyección ciega del Top-K)"))
    salida_naive = ejecutar_consulta(
        prompt=f"Contexto:\n{contexto}\n\nPregunta: {q['texto']}",
        system_prompt="Eres un asistente de seguros. Responde basandote en el contexto provisto.",
        temperature=0.1, contexto_referencia=contexto)
    aud_naive = auditar_respuesta(salida_naive["texto"], contexto, FICHA_E4)

    # --- Vía 2: CRAG, evalúa antes de generar ---
    display(Markdown(f"### 🛡️ Corrective RAG (umbral {UMBRAL_E4:.2f})"))
    if aprobados:
        ctx_ok = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in aprobados)
        salida_crag = ejecutar_consulta(
            prompt=f"Contexto convalidado:\n{ctx_ok}\n\nPregunta: {q['texto']}",
            system_prompt="Responde basandote exclusivamente en los fragmentos convalidados.",
            temperature=0.1, contexto_referencia=ctx_ok)
        aud_crag = auditar_respuesta(salida_crag["texto"], ctx_ok, FICHA_E4)
        ahorro = 0
    else:
        try:
            ahorro = len(llm.tokenize(contexto.encode("utf-8")))
        except Exception:
            ahorro = -1
        # Misma tabla de telemetria que ejecutar_consulta, para que la via bloqueada sea
        # comparable columna a columna contra la via que si genero.
        display(Markdown(
            f"{MSG_CRAG}\n\n---\n\n"
            "| ⏱️ TTFT | 🏁 EERL | ⚡ TPOT | 🧮 Tokens |\n"
            "| ---: | ---: | ---: | ---: |\n"
            f"| — | 0.00 s | — | {ahorro} no enviados |\n\n"
            "<sub>Llamada abortada por el filtro: no hubo generación, así que no hay latencia "
            "ni tokens de salida que medir.</sub>"))
        aud_crag = {"exito": True, "cifras_inventadas": [], "cifras_prohibidas": [],
                    "citas_falsas": [], "se_abstuvo": True, "debe_abstenerse": True,
                    "trazable": False, "terminos_faltantes": []}

    resumen_e4.append({"q": q, "sim": sim_max, "filtrada": not aprobados,
                       "naive": aud_naive, "crag": aud_crag, "ahorro": ahorro})

tabla = f"""### 🎯 El filtro frena una pregunta y deja pasar la otra

Las dos preguntas no tienen respuesta en el corpus. Lo único que las distingue es cuánto **se parecen** al manual — y el umbral de {UMBRAL_E4:.2f} sólo puede medir eso.

| Pregunta | Similitud máx. | ¿CRAG la bloquea? | ⚠️ Naive RAG | 🛡️ CRAG |
| :--- | ---: | :---: | :--- | :--- |
"""
for r in resumen_e4:
    tabla += (f"| **{r['q']['id']}** · {r['q']['clase']} | `{r['sim']:.4f}` | "
              f"{'🟢 SÍ' if r['filtrada'] else '🔴 NO'} | {semaforo(r['naive'])} | "
              f"{'🟢 CORRECTA' if r['crag']['exito'] else semaforo(r['crag'])} |\n")

tabla += """
> 🎯 **La lección incómoda del experimento:** CRAG no verifica si la respuesta existe. Verifica si los
> documentos **se parecen** a la pregunta. Por eso frena en seco la pregunta A (otro dominio,
> similitud baja) y deja pasar la pregunta B (mismo dominio, similitud alta) aunque el dato pedido
> no figure en ningún artículo.
>
> 🧪 **Y fijate qué hizo el Naive RAG con la pregunta B**, que es la misma trampa del Experimento 3 vista desde acá. Su semáforo cambia de corrida en corrida: unas veces dice `🔴 ALUCINACION POR CONTAMINACION` —se trae la franquicia de 90 USD, que es la de **cristales**, y la ofrece como si fuera la del robo— y otras dice `🔴 NO SE ABSTUVO`, cuando reconoce que el dato no está pero igual cierra con una conclusión propia. Es el mismo defecto con dos caras, y ninguna de las dos la frena el umbral.
>
> 🧱 **Conclusión de arquitectura — la defensa es en capas, no una sola muralla:**
> el umbral de CRAG detiene lo que está *lejos*; la **regla de abstención del Experimento 3** detiene lo
> que está *cerca pero ausente*. Ninguna de las dos alcanza sola, y por eso los sistemas de
> producción en industrias reguladas usan las dos al mismo tiempo.
"""
display(Markdown(tabla))

In [ ]:
#@title 🚀 4.B — Barrido del umbral: el filtro que protege también puede estrangular al sistema { display-mode: "form" }
# ¿Existe "el" umbral correcto? Lo medimos contra un set etiquetado: seis consultas de las que
# ya sabemos la respuesta correcta — tres que el manual SÍ cubre y tres que no.

SET_EVALUACION = [
    # --- deben PASAR: el corpus las responde ---
    ("¿Cuántas horas tengo para denunciar un choque en la autopista?",        True),
    ("Quiero cancelar la póliza antes de tiempo, ¿qué me descuentan?",        True),
    ("¿Hasta qué monto cubre la responsabilidad civil frente a terceros?",    True),
    # --- deben BLOQUEARSE: están fuera del corpus ---
    ("¿Cuáles son los requisitos para un crédito hipotecario a 20 años?",     False),
    ("¿Cómo tramito el pasaporte y la visa de turista para Europa?",          False),
    ("¿Qué rendimiento anual paga hoy un plazo fijo en pesos?",               False),
]

sims = [(indice.denso(q, top_k=1), cubierta) for q, cubierta in SET_EVALUACION]
sims = [(rs[0]["score"] if rs else 0.0, cubierta) for rs, cubierta in sims]

filas, mejor = [], None
for u in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    vp = sum(1 for s, c in sims if c and s >= u)          # legítima que pasa  ✔
    fn = sum(1 for s, c in sims if c and s < u)           # legítima bloqueada ✘ (sistema inútil)
    vn = sum(1 for s, c in sims if not c and s < u)       # ruido bloqueado    ✔
    fp = sum(1 for s, c in sims if not c and s >= u)      # ruido que pasa     ✘ (alucinación)
    acc = (vp + vn) / len(sims)
    filas.append((u, vp, fn, vn, fp, acc))
    if mejor is None or acc > mejor[5]:
        mejor = (u, vp, fn, vn, fp, acc)

txt = """### 🎯 No existe "el" umbral correcto: existe el que elegís

Seis consultas de respuesta conocida -tres que el manual cubre y tres que no- evaluadas contra once umbrales. Cada fila es una política distinta, con sus dos formas de equivocarse.

| Umbral | ✔️ Legítimas que pasan | ❌ Legítimas bloqueadas | ✔️ Ruido bloqueado | ❌ Ruido que pasa | Exactitud |
| ---: | :---: | :---: | :---: | :---: | :--- |
"""
for u, vp, fn, vn, fp, acc in filas:
    marca = " ⬅️ **óptimo**" if u == mejor[0] else ""
    barra = "█" * round(acc * 10) + "░" * (10 - round(acc * 10))
    txt += f"| `{u:.2f}` | {vp}/3 | {fn}/3 | {vn}/3 | {fp}/3 | `{barra}` {acc:.0%}{marca} |\n"

txt += ("\n#### 🔎 De dónde sale cada número\n\n"
        "La similitud máxima de cada consulta del set: es el dato crudo sobre el que se calcula toda la curva de arriba.\n\n"
        "| Consulta | ¿La cubre el manual? | Similitud máx. |\n| :--- | :---: | ---: |\n")
for (q, cubierta), (s, _) in zip(SET_EVALUACION, sims):
    txt += f"| *{q}* | {'🟢 sí' if cubierta else '🔴 no'} | `{s:.4f}` |\n"

if mejor[5] >= 0.999:
    lectura_zona = (f"**separa ambas poblaciones sin un solo error** (exactitud 100%). En este "
                    f"corpus, con este modelo, las franjas de similitud de lo legítimo y lo ajeno "
                    f"no se tocan.")
else:
    fallo = "un falso negativo (rechaza una consulta legítima)" if mejor[2] > 0 else "un falso positivo (deja pasar ruido)"
    lectura_zona = (f"**minimiza los errores, pero no los elimina** (mejor exactitud alcanzable: "
                    f"{mejor[5]:.0%}, con {fallo}). Las franjas de similitud de lo legítimo "
                    f"(`{min(s for s,c in sims if c):.2f}`–`{max(s for s,c in sims if c):.2f}`) y de "
                    f"lo ajeno (`{min(s for s,c in sims if not c):.2f}`–`{max(s for s,c in sims if not c):.2f}`) "
                    f"se **solapan**: ningún umbral único es perfecto sobre este set.")

txt += f"""
> 📌 **El umbral no es un ajuste técnico: es una decisión de negocio.**
>
> * **Umbral bajo (0.20–0.30):** no bloquea nada. El sistema contesta todo, incluso lo que no sabe.
>   Máxima cobertura, máxima exposición a alucinaciones.
> * **Umbral alto (0.65–0.70):** bloquea hasta las consultas legítimas. El sistema se vuelve un
>   *«no poseo información»* permanente: técnicamente impecable, comercialmente inservible.
> * **Zona útil (≈ {mejor[0]:.2f} en este corpus):** {lectura_zona} Y este número **no es
>   universal** — depende del modelo de embeddings, del idioma y del corpus. Se calibra midiendo,
>   como acabamos de hacer, nunca copiándolo de un tutorial.
>
> ⚖️ **Cómo se elige en la práctica:** en un buscador interno de documentación, un falso negativo
> apenas molesta. En un bot que le informa coberturas a un asegurado, un falso positivo puede
> terminar en un reclamo judicial. El umbral se corre hacia el error que la organización
> **puede permitirse pagar** — y si ningún umbral los elimina a todos, la conclusión no es buscar
> un número mejor: es agregar otra capa de defensa (la regla de abstención del Experimento 3).
"""
display(Markdown(txt))

In [ ]:
#@title 🧪 4.C — Exploración libre: mové el umbral y mirá la bifurcación en vivo { display-mode: "form" }
import ipywidgets as widgets

preg4_w = widgets.Dropdown(
    options=[q["texto"] for q in PREGUNTAS_E4] + [q for q, _ in SET_EVALUACION],
    value=PREGUNTAS_E4[1]["texto"], description="Consulta:",
    style={'description_width': 'initial'}, layout=widgets.Layout(width='90%'))
umbral_w = widgets.FloatSlider(value=UMBRAL_E4, min=0.10, max=0.80, step=0.01,
                               description="Umbral CRAG:", readout_format='.2f',
                               style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
gen_w = widgets.Checkbox(value=False, description="Además, generar la respuesta con el LLM (más lento)")
out4 = widgets.Output()

def _evaluar(_=None):
    with out4:
        out4.clear_output(wait=True)
        q, u = preg4_w.value, umbral_w.value
        rec = indice.denso(q, top_k=3)
        t = ("### 🛡️ El evaluador, paso a paso\n\n"
             "Lo que CRAG mira antes de decidir si vale la pena llamar al modelo.\n\n"
             "| Chunk | Artículo | Similitud | Umbral | Dictamen |\n| :--- | :--- | ---: | ---: | :--- |\n")
        aprobados = []
        for r in rec:
            ok = r["score"] >= u
            if ok:
                aprobados.append(r)
            t += (f"| `{r['chunk']['id']}` | `{r['chunk']['articulo']}` | `{r['score']:.4f}` | "
                  f"`{u:.2f}` | {'🟢 relevante' if ok else '🔴 descartado'} |\n")
        if aprobados:
            t += (f"\n**Ruta:** 🟢 vía confiable — {len(aprobados)} de {len(rec)} fragmentos "
                  f"superaron el umbral y van al generador.\n")
        else:
            try:
                ahorro = len(llm.tokenize("\n\n".join(r['chunk']['texto'] for r in rec).encode("utf-8")))
            except Exception:
                ahorro = -1
            t += (f"\n**Ruta:** 🛑 cortocircuito de seguridad — nada superó el umbral. "
                  f"Se abortó la llamada al LLM y se ahorraron **{ahorro} tokens** de contexto.\n")
        display(Markdown(t))
        if gen_w.value and aprobados:
            ctx = "\n\n".join(f"[{r['chunk']['id']}]: {r['chunk']['texto']}" for r in aprobados)
            ejecutar_consulta(prompt=f"Contexto convalidado:\n{ctx}\n\nPregunta: {q}",
                              system_prompt="Responde basandote exclusivamente en los fragmentos convalidados.",
                              temperature=0.1, contexto_referencia=ctx)

for w in (preg4_w, umbral_w):
    w.observe(_evaluar, names="value")
display(widgets.VBox([preg4_w, umbral_w, gen_w, out4]))
_evaluar()

### 📝 Registro de Análisis — Experimento 4 (Entregable TP2)

1. **Dónde funciona y dónde no.** Transcribí la similitud máxima de la pregunta A y de la B. Explicá con esos números por qué CRAG bloquea una y deja pasar la otra, y por qué **no se trata de un error de implementación** sino de una limitación conceptual del patrón.
2. **Calibración como decisión de negocio.** Del barrido de `4.B`, indicá un umbral demasiado permisivo y uno demasiado estricto, con las consecuencias de cada uno. Después elegí un valor y justificalo para dos escenarios opuestos: un buscador interno de documentación técnica y un bot que informa coberturas a asegurados.
3. **Economía de la abstención.** Cuando CRAG corta la llamada, el ahorro es de tokens y de latencia. Pero también hay un costo: consultas legítimas rechazadas. ¿Cómo instrumentarías la medición de ese costo en producción? ¿Qué métrica pondrías en un tablero de monitoreo?
4. **Defensa en capas.** El Experimento 3 aportó la regla de abstención y el Experimento 4 el filtro por umbral. Explicá qué tipo de falla detiene cada uno y por qué un sistema en una industria regulada necesita **ambos** mecanismos y no puede elegir uno solo.

> ✍️ **Tu análisis aquí:** *(Escribí acá tus observaciones y conclusiones para la entrega del TP2)*

## 🏁 Cierre del Laboratorio y Retrospectiva

---
### 🧠 Matriz de decisión: qué usar y cuándo

| Desafío operativo | Alternativa arquitectónica | Parámetros de referencia | Lo que resuelve | Lo que **no** resuelve |
| :--- | :--- | :--- | :--- | :--- |
| Contratos con reglas y excepciones encadenadas | Chunking recursivo con solape | `size 400–600` · `overlap 15–20%` | Que la excepción viaje junto a su regla | Documentos con tablas o anexos referenciados a distancia |
| Consultas informales, sinónimos, jerga | Búsqueda densa (*embeddings*) | `Top-K 2–4` · modelo multilingüe | Salvar la brecha entre el habla del usuario y el lenguaje del manual | Códigos, siglas, números de artículo |
| Códigos, SKUs, números de cláusula | Búsqueda dispersa (BM25) | stopwords del idioma · IDF | Coincidencia exacta imbatible | Preguntas parafraseadas |
| Producción, tráfico real y heterogéneo | **Búsqueda híbrida (RRF)** | fusión por ranking · `k≈60` | Cubrir los puntos ciegos de ambos motores | El costo de mantener dos índices |
| Riesgo de litigio o auditoría | RAG fundamentado con citas | `Temp 0.0–0.1` · citas obligatorias | Trazabilidad y abstención ante el vacío | Que el recuperador haya traído el fragmento correcto |
| Consultas fuera de dominio | Corrective RAG (umbral) | umbral **calibrado sobre tu corpus** | Frenar lo que está temáticamente lejos | Lo que está cerca pero cuyo dato no existe |

> 🧱 **La síntesis del laboratorio:** ninguna de estas piezas es suficiente por sí sola. Un índice impecable no salva a un prompt sin reglas; un prompt riguroso no inventa la cláusula que el chunking perdió; y un umbral bien calibrado no detecta la pregunta cuya respuesta simplemente no está escrita. **La confiabilidad es la composición de las cuatro capas**, y cada una se calibra midiendo, no estimando.

---
### 📊 Las cuatro dimensiones de evaluación (marco RAGAS)
Lo que fuimos midiendo experimento por experimento tiene nombre propio en la literatura:

| Dimensión | Qué evalúa | Dónde la medimos |
| :--- | :--- | :--- |
| **Context Relevance** | Si lo recuperado contiene lo necesario | Experimento 2 — marcador contra *ground truth* |
| **Context Precision** | Si lo útil quedó en los primeros puestos | Experimento 2 — posición en el Top-K |
| **Faithfulness / Groundedness** | Si cada afirmación se sostiene en la evidencia | Experimento 3 — detector de cifras inventadas |
| **Answer Relevance** | Si la respuesta contesta lo que se preguntó | Experimento 3 — abstención correcta vs. indebida |

Y en el plano operativo: **TTFT** (percepción de lentitud), **TPOT** (velocidad de generación), **EERL** (latencia total) y la **economía de tokens**, visibles en cada panel de telemetría.

---
### ✅ Checklist de entrega del TP2 (Parte 1)
* [ ] **Entorno validado:** celdas `0.1`, `0.2` y `0.3` ejecutadas sobre GPU T4.
* [ ] **Experimento 1:** ejecutaste `1.A`, `1.B` y `1.C`, y registraste el valor de solape donde el índice llega a `4/4`.
* [ ] **Experimento 2:** ejecutaste `2.A` y transcribiste el marcador de los tres motores.
* [ ] **Experimento 3:** ejecutaste `3.A` y analizaste la matriz completa de 3 arquitecturas × 2 preguntas.
* [ ] **Experimento 4:** ejecutaste `4.A` y `4.B`, y elegiste un umbral justificándolo para dos escenarios.
* [ ] **Las cuatro reflexiones** completadas en los recuadros `✍️ Tu análisis aquí:`.
* [ ] **Respaldo:** `Archivo` → `Guardar una copia en Drive`, o descarga del `.ipynb`.

---
### 🌉 Puente hacia la Clase 4: Herramientas, Validación y Agentes
En esta sesión conectamos un LLM a un repositorio documental **estático** y lo obligamos a responder sólo con evidencia citada. Pero un manual no puede responderlo todo.

¿Qué pasa si el asegurado pregunta **cuánto le corresponde exactamente** por su siniestro? Eso requiere **calcular** sobre el valor de su unidad, aplicar la depreciación del 25% anual según la antigüedad y descontar la franquicia. Una base vectorial no hace aritmética. ¿Y si necesita el tipo de cambio de hoy, o el saldo real de su cuenta en una base transaccional? **Ningún documento indexado contiene un dato que cambia cada segundo.**

En la **Clase 4** incorporamos el paradigma de **Tool Calling**: enseñarle al modelo a reconocer cuándo debe delegar una operación a una calculadora, una API o una consulta SQL — y a validar lo que recibe antes de usarlo. Es la pieza que convierte a este sistema de consulta en un asistente corporativo capaz de actuar.